# Data Analysis & Visualization Notebook

## Overview

Brief description of your analysis objectives and the data sources
you’ll be working with.

## Setup & Dependencies


In [18]:
import os, sys, json, datetime, re  # Provides OS-dependent functionality, system-specific parameters, JSON handling, and date/time manipulation
import pandas as pd             # Provides data structures and data analysis tools
import numpy as np              # Supports large, multi-dimensional arrays and matrices
import requests
import time
from tqdm import tqdm
import glob as glob

#thi data contants
from cprl_functions.defined_functions import *
from cprl_functions.state_capture import *
from cprl_functions.text_printing import bordered
from cprl_functions.data_packet_defs import *

#Import Data
###################
import data_sources.data_collection.pull_data as data_pull
from graphs.viz_graphs_template import *


# Create folder if needed
graphs_dir = r"C:\Users\clutz\THE HUNT INSTITUTE\The Hunt Institute Team Site - Documents\Policy Team\Data Packets\K-12\graphs"




In [27]:
#controls
save_it = True
show_it = False
just_one = False

In [20]:
state_abbreviations_priority

['AZ',
 'CT',
 'DC',
 'DE',
 'GA',
 'ID',
 'IL',
 'IN',
 'IA',
 'KS',
 'KY',
 'MD',
 'MA',
 'MI',
 'MN',
 'MO',
 'NE',
 'NJ',
 'NM',
 'NY',
 'NC',
 'ND',
 'OH',
 'OK',
 'OR',
 'RI',
 'SC',
 'TN',
 'TX',
 'VT',
 'VA',
 'WA',
 'WV',
 'WY']

In [21]:
graphs_dir = r'C:\Users\clutz\THE HUNT INSTITUTE\The Hunt Institute Team Site - Documents\Policy Team\Data\Data Packets\K-12\graphs'
def save_to_folder(fig,filename, g_width,g_height, state):
    # Create folder if needed
    output_folder = os.path.join(graphs_dir,state)
    os.makedirs(output_folder, exist_ok=True)

    # Save the file
    fig.write_image(
        
        os.path.join(output_folder,filename),
        width=g_width,
        height=g_height,
        scale=2  # For higher resolution (2x default)
    )


#### helper functions

In [22]:
def get_highlight_color(x,state):
    if x==state:
        return hunt_light_purple
    else:
        return hunt_darkgray
    

In [23]:
# data_directory = Path(r'C:\Users\clutz\THE HUNT INSTITUTE\The Hunt Institute Team Site - Documents\Policy Team\Data Packets\k12\data')
# def get_files():
#     file_dict = {}
#     for file in data_directory.iterdir():
#         print(file.name)
#         if 'naep' in file.name.lower():
#             naep_file = data_directory / file.name
#             file_dict['naep'] = naep_file
#         elif 'data_collection' in file.name.lower():
#             data_collection_file = data_directory / file.name
#             file_dict['data_collection'] = data_collection_file
        

#     return file_dict


        



# with open(data_directory / naep_file, 'r') as file:
#     excel_file = pd.ExcelFile(file,engine='calamine')
#     for sheet in excel_file.sheet_names:
#         if '- 2024' in sheet:
#             print(sheet) 

# excel_file = pd.ExcelFile(data_directory / data_collection_file, engine='calamine')
# for sheet in excel_file.sheet_names:
#     print(sheet)
#     if 'merge' in sheet.lower():
#         merge_tags = excel_file.parse(sheet_name=sheet)
#         break

# print(merge_tags.to_string())



# glob_pat = os.path.join(filepath, 'NAEP*.xlsx')
# files = glob.glob(glob_pat)
# print(merge_tags)
# for file in filepath:
# pd.ExcelFile()

# Merge File

In [14]:
merge_dir = Path(r'C:\Users\clutz\THE HUNT INSTITUTE\The Hunt Institute Team Site - Documents\Policy Team\Data\Data Packets\K-12\merge_files')

merge_dfs = {}
for root, dir, files in os.walk(merge_dir):
    if len(files)!=0:
        # print(root)
        # print(files)
        file = files[0]
        print(file)
        df = pd.read_csv(os.path.join(root,file)).reset_index(drop=True)
        if any('Unnamed: 0' in str(x) for x in df.columns):
            df = df.drop(columns = ['Unnamed: 0'])
        

        print(df.to_string())
        type =root.split('\\')[-1]
        # print(type)
        merge_dfs[type] = df


merge.csv
   state_abrv              00 State  Number of Public Schools  Number of Public School Districts Per-Pupil Expenditure  State Rank - Per-Pupil Expenditure  Student-Teacher Ratio Years - Student-Teacher Ratio  State Rank - Student-Teacher Ratio NAEP 4th Grade Math Proficiency %  State Rank - NAEP 4th Grade Math Proficiency % NAEP 4th Grade Reading Proficiency %  State Rank - NAEP 4th Grade Reading Proficiency % NAEP 8th Grade Math Proficiency %  State Rank - NAEP 8th Grade Math Proficiency % NAEP 8th Grade Reading Proficiency %  State Rank - NAEP 8th Grade Reading Proficiency % Public HS Graduation Rate  State Rank - Public HS Graduation Rate Years - K-12 Enrollment by Race Graphic Years - K-12 Enrollment by SES Graphic                                                                                                                                                                                                                                                                      

In [16]:

# merge_dfs.get('graphs')
# pd.merge([merge_dfs.get('text'), merge_dfs.get('graphs')], on=)
# merge_file = pd.concat([merge_dfs.get('text'), merge_dfs.get('graphs')], axis=1).reset_index()
merge_file = merge_dfs.get('text').merge(merge_dfs.get('graphs'), on='state_abrv')

print(merge_file.to_string())
merge_file.to_csv(os.path.join(merge_dir,'merge.csv'), index=False)


   state_abrv              00 State  Number of Public Schools  Number of Public School Districts Per-Pupil Expenditure  State Rank - Per-Pupil Expenditure  Student-Teacher Ratio Years - Student-Teacher Ratio  State Rank - Student-Teacher Ratio NAEP 4th Grade Math Proficiency %  State Rank - NAEP 4th Grade Math Proficiency % NAEP 4th Grade Reading Proficiency %  State Rank - NAEP 4th Grade Reading Proficiency % NAEP 8th Grade Math Proficiency %  State Rank - NAEP 8th Grade Math Proficiency % NAEP 8th Grade Reading Proficiency %  State Rank - NAEP 8th Grade Reading Proficiency % Public HS Graduation Rate  State Rank - Public HS Graduation Rate Years - K-12 Enrollment by Race Graphic Years - K-12 Enrollment by SES Graphic                                                                                                                                                                                                                                                                                

# Graphs

## Page 1

In [24]:
# pull in enrollment data
enroll_total_df = data_pull.total_enroll_data_clean()
enroll_re_df = data_pull.re_enroll_data_clean(enroll_total_df)

# print(enroll_total_df.to_string())
# print(enroll_re_df.to_string())

Index(['state_abrv', 'fy', 'year', 'total_enrollment',
       'american_indian_alaska_native', 'hispanic', 'black', 'white',
       'two_or_more', 'asian_pacific_islander'],
      dtype='object')


### Total K-12 Enrollment

In [31]:
#creates table 1.1
from graphs.viz_graphs_template import *
import plotly.graph_objects as go
import plotly.express as px



g_width = 1102
g_height = 915
offset_control = .1
multiplier = 1
filename = f'1.1.png'
enroll_total_df = data_pull.total_enroll_data_clean()

for jur in state_abbreviations_priority:
    if jur == 'US':
        continue
    print(jur)
    #filter for only state vals
    result = enroll_total_df[enroll_total_df['state_abrv']==jur]
    # print(result.to_string())
    
    #call graph
    fig = graph_1_1(result['year'], result['total_enrollment'])
    # print(type(fig))
    
    #position calculations for text annotations
    ymin = min(result['total_enrollment'])
    ymax = max(result['total_enrollment'])
    yrange = ymax - ymin
    offset = yrange * offset_control  # 5% of the total vertical range
    range_offset = yrange * (offset_control*2)

    enrollment_text = result['total_enrollment'].to_list()
    text_y_positions = []  # Store y-positions for text    
    for i, x in enumerate(enrollment_text):
        if i == 0:
            # First element - compare with next
            next_val = enrollment_text[i+1]
            if next_val > x:
                value = x - offset  # Position below
            else:
                value = x + offset  # Position above
                
        elif i < len(enrollment_text) - 1:  # Fixed: was missing "- 1"
            # Middle elements - has both previous and next
            next_val = enrollment_text[i+1]
            last = enrollment_text[i-1]
            
            # Calculate slopes (direction of change)
            slope_to = x - last  # Positive if increasing
            slope_from = next_val - x  # Positive if will increase
            
            if slope_to > 0 and slope_from > 0:
                # Going up and will continue up
                value = x + offset
            elif slope_to < 0 and slope_from < 0:
                # Going down and will continue down
                value = x + offset
            elif slope_to > 0 and slope_from < 0:
                # Peak (was going up, now going down)
                value = x + (multiplier * offset)
            elif slope_to < 0 and slope_from > 0:
                # Valley (was going down, now going up)
                value = x - offset
            else:
                # Flat
                value = x + offset
                
        else:
            # Last element - compare with previous
            last = enrollment_text[i-1]
            if x > last:
                value = x + offset  # Was increasing, put above
            else:
                value = x - offset  # Was decreasing, put below
        
        text_y_positions.append(value)
    
    # Add text labels
    fig.add_trace(go.Scatter(
        x=result['year'].to_list(),
        y=text_y_positions,  # Use calculated positions
        mode='text',
        text=[f"{int(val):,}" for val in enrollment_text],  # Format numbers with commas
        textposition='middle center',
        showlegend=False
    ))
    fig.update_yaxes(
        showticklabels=False,  # Hide tick labels
        showgrid=False,        # Hide gridlines
        zeroline=False,         # Hide zero line
        range=[ymin - range_offset, ymax + range_offset]
    )    
    
    if show_it:
        fig.show()
        if just_one:
            break
    if save_it:
        save_to_folder(fig,filename,g_width,g_height,jur)

    
    # break

AZ


C:\Users\clutz\AppData\Local\Temp\ipykernel_15552\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




CT
DC
DE
GA
ID
IL
IN
IA
KS
KY
MD
MA
MI
MN
MO
NE
NJ
NM
NY
NC
ND
OH
OK
OR
RI
SC
TN
TX
VT
VA
WA
WV
WY


#### Enrollment by Race and Ethnicity 

In [30]:
#creates table 1.2 (example of 100% bar)
from graphs.viz_graphs_template import *
import plotly.graph_objects as go
import plotly.express as px



#graph settings

aspect_ratio = 7
graph_h = 200
graph_wd = graph_h*aspect_ratio

offset = 1500
multiplier = 1.5
filename = f'1.2.png'
enroll_re_df = data_pull.re_enroll_data_clean(enroll_total_df)


us_values = enroll_re_df[enroll_re_df['state_abrv']=="US"]

us_values = us_values.sort_values('year', ascending=False).reset_index(drop=True)
print(us_values[us_values['fy']==us_values['fy'].max()].to_string())
# us_total
for jur in state_abbreviations_priority:
    if jur == 'US':
        continue
    # print(jur)
    
    #fetch for state
    result = enroll_re_df[enroll_re_df['state_abrv']==jur]
    # print(result)
    
    us_series = us_values[us_values['fy']==us_values['fy'].max()]
    
    #set up/cleaning
    series_dict = {'state':result,'us':us_series}
    
    
    if all(len(x)==1 for x in series_dict.values()):
        # print('good to keep going')
        # print(series_dict.get('state').columns)
        categories = series_dict.get('state').columns[5:]
        state_vals = series_dict.get('state').iloc[0,5:].to_list()
        us_vals = series_dict.get('us').iloc[0,5:].to_list()
        both_values_dict = {"us":us_vals, 'state':state_vals}
        for k,v in both_values_dict.items():
            new_values = [int(x) for x in v]
            both_values_dict[k] = new_values

        
        # print(state_percents)
        # us_values = series_dict.get('state').columns[5:]
        categories = [x.replace('_'," ").title() for x in categories]
        
        
        
        fig = graph_1_2(categories, state_vals, us_vals, jur, graph_wd, graph_h)
        # save_to_folder(fig,filename,g_width,g_height,state)
        if show_it:
            fig.show()
            if just_one:
                break
        if save_it:
            save_to_folder(fig,filename,graph_wd,graph_h,jur)


    else:
        for k,x in series_dict.items():
            print('not good')
    # print(result['fy'].max())
    


Index(['state_abrv', 'fy', 'year', 'total_enrollment',
       'american_indian_alaska_native', 'hispanic', 'black', 'white',
       'two_or_more', 'asian_pacific_islander'],
      dtype='object')
  state_abrv     year  fy  total_enrollment american_indian_alaska_native  asian_pacific_islander    hispanic      black       white two_or_more
0         US  2023-24  24        49404386.0                      442209.0                 2940883  14532747.0  7352891.0  21592113.0   2522052.0


C:\Users\clutz\AppData\Local\Temp\ipykernel_15552\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


C:\Users\clutz\AppData\Local\Temp\ipykernel_15552\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


C:\Users\clutz\AppData\Local\Temp\ipykernel_15552\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


C:\Users\clutz\AppData\Local\Temp\ipykernel_15552\2484680588.py:

#### K-12 Enrollment, by Socioeconomic Status

In [29]:
#creates table 1.3 (example of 100% bar)
from graphs.viz_graphs_template import *
import plotly.graph_objects as go
import plotly.express as px


g_width = 1142
g_height = 541
filename = f'1.3.png'
ses_data = data_pull.get_collected_data(metric = 'enr_ses')
ses_data.columns = [cleaning_col(x) for x in ses_data.columns]
# print(ses_data.to_string())
# data_pull.get_collected_data(get_sheets=True)

#graph settings
aspect_ratio = 7
graph_h = 200
graph_wd = graph_h*aspect_ratio

offset = 1500
multiplier = 1.5

#preset values before loop
categories = ['Economically Disadvantaged', 'Not Economically Disadvantaged']
us_values = ses_data[ses_data['state_abrv']=="US"]
# print('US only')
# print(us_values.to_string())

us_values = us_values.sort_values('year', ascending=False).reset_index(drop=True)
us_dict = us_values.to_dict(orient='records')[0]

# us_total
for jur in state_abbreviations_priority:
    if jur == 'US':
        continue
    # print(jur)
    state_name = state_ref_r.get(jur)
    #fetch for state
    result = ses_data[ses_data['state_abrv']==jur]
    result = result.loc[:,['state_abrv','year','percent_economically_disadvantaged']]
    
    # result = result[result['year']==result['year'].max()]
    res_dict = result.to_dict(orient='records')[0]
    # print(res_dict)
    # print(type(us_values))
    # print(us_values)
    dfs = {f'{jur}':result, "US":us_values}
    # concat_dfs = []

  
    geos = [state_name,'United States'] 
    
    fig = graph_1_3(res_dict, us_dict, jur, graph_wd, graph_h)
    save_to_folder(fig,filename,g_width,g_height,jur)
    # break

    # graph_1_2(state_values, us_values)
        # print(categories)

    # else:
    #     for k,x in series_dict.items():
    #         print('not good')
    # print(result['fy'].max())
    


C:\Users\clutz\AppData\Local\Temp\ipykernel_15552\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


C:\Users\clutz\AppData\Local\Temp\ipykernel_15552\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


C:\Users\clutz\AppData\Local\Temp\ipykernel_15552\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


C:\Users\clutz\AppData\Local\Temp\ipykernel_15552\2484680588.py:

### 1.4: K-12 Enrollment, by Locale


## Page 2

### STATE ASSEMEMNT RESULTS

In [10]:
import pandas as pd
import numpy as np

def reshape_education_data(df):
    """
    Reshapes education data from wide format to long format.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Input dataframe with education data in wide format
        
    Returns:
    --------
    pandas.DataFrame
        Reshaped dataframe with columns: state_abrv, 2025_available, year, grade, subject, value
    """
    
    # Create a list to store all rows
    rows = []
    
    # Define the column mapping for years and subjects
    # Columns 5-7: 4th grade Reading (2023, 2024, 2025)
    # Columns 8-10: 4th grade Math (2023, 2024, 2025)
    # Columns 11-13: 8th grade Reading (2023, 2024, 2025)
    # Columns 14-16: 8th grade Math (2023, 2024, 2025)
    
    col_mapping = [
        (5, '2023', '4th grade', 'Reading'),
        (6, '2024', '4th grade', 'Reading'),
        (7, '2025', '4th grade', 'Reading'),
        (8, '2023', '4th grade', 'Math'),
        (9, '2024', '4th grade', 'Math'),
        (10, '2025', '4th grade', 'Math'),
        (11, '2023', '8th grade', 'Reading'),
        (12, '2024', '8th grade', 'Reading'),
        (13, '2025', '8th grade', 'Reading'),
        (14, '2023', '8th grade', 'Math'),
        (15, '2024', '8th grade', 'Math'),
        (16, '2025', '8th grade', 'Math'),
    ]
    
    # Iterate through each state (starting from row 3)
    for i in range(3, len(df)):
        state = df.iloc[i, 1]
        unavailable_flag = df.iloc[i, 4]
        
        # Determine if 2025 data is available
        # If the flag is True, then 2025 is unavailable, otherwise it's available
        data_2025_available = 'No' if unavailable_flag == True or (pd.notna(unavailable_flag) and str(unavailable_flag).strip().lower() == 'true') else 'Yes'
        
        # Iterate through each column mapping
        for col_idx, year, grade, subject in col_mapping:
            value = df.iloc[i, col_idx]
            
            # Skip if value is NaN or 0 (which seems to indicate missing data in your dataset)
            if pd.notna(value) and value != 0:
                rows.append({
                    'state_abrv': state,
                    '2025_available': data_2025_available,
                    'year': year,
                    'grade': grade,
                    'subject': subject,
                    'value': value
                })
    
    # Create the final dataframe
    result_df = pd.DataFrame(rows)
    
    # Sort by state, grade, subject, and year for better organization
    result_df = result_df.sort_values(['state_abrv', 'grade', 'subject', 'year']).reset_index(drop=True)
    
    return result_df


# Example usage:
# result_df = reshape_education_data(df)
# print(result_df.head(20))
# print(f"\nTotal rows: {len(result_df)}")
# print(f"\nStates with 2025 data available: {result_df[result_df['2025_available'] == 'Yes']['state_abrv'].nunique()}")
# print(f"States with 2025 data unavailable: {result_df[result_df['2025_available'] == 'No']['state_abrv'].nunique()}")
# result_df.to_csv('education_data_long_format.csv', index=False)

In [11]:
# GRAPHS for STATE ASSESSMENT RESULTS
g_width = 1352
g_height = 768

state_assess_res = data_pull.get_collected_data(metric='state_assess_prof', no_header=True)
state_assess_res = state_assess_res.iloc[:,:17]
# cols = list(state_assess_res.columns)
# state_assess_res.columns = ['priority']+ cols[1:]
state_assess_res = state_assess_res[state_assess_res[0]!='x']


state_assess_res = reshape_education_data(state_assess_res)
state_assess_res = state_assess_res.sort_values(by=['state_abrv'])
state_assess_res = state_assess_res.sort_values(by=['grade','year','subject'], ascending=False).reset_index(drop=True)
grades = sorted(get_col_uniq_vals(state_assess_res['grade']))

# for state in state_abbreviations_priority:
#     if state == 'US':
#         continue
#     results = state_assess_res[state_assess_res['state_abrv']==state]

#     for grade in grades:
#         print(grade)
#         grad_res = results[results['grade']==grade]
#         print(grad_res.to_string())
#         fig = graph_2_state_assess_graph(grad_res, state)
#         fig.show()
for state in state_abbreviations_priority:
    if state == 'US':
        continue
    results = state_assess_res[state_assess_res['state_abrv']==state]

    for i,grade in enumerate(grades):
        print(grade)
        grad_res = results[results['grade']==grade]
        # Sort by year and subject to ensure consistent ordering
        grad_res = grad_res.sort_values(by=['year', 'subject']).reset_index(drop=True)
        print(grad_res.to_string())
        fig = graph_2_state_assess_graph(grad_res, state)
        fig.update_layout(
            font=dict(
                family='Lato',
                size=20,
                color=hunt_darkgray
            ),
            width=g_width,
            height = g_height
            )
        # fig.update_xaxes(
        #     tickvals=['2022-2023', '2023-2024', '2024-2025'],
        # )
                    

        fig.update_yaxes(
            visible=False)
        if show_it:
            fig.show()
            if just_one:
                break
        if save_it:
            save_to_folder(fig,f'2.{i+1}.png',g_width,g_height,state)
            

4th grade
  state_abrv 2025_available  year      grade  subject  value
0         AZ             No  2023  4th grade     Math   39.0
1         AZ             No  2023  4th grade  Reading   45.0
2         AZ             No  2024  4th grade     Math   36.0
3         AZ             No  2024  4th grade  Reading   46.0
4         AZ             No  2025  4th grade     Math   40.0
5         AZ             No  2025  4th grade  Reading   46.0


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




8th grade
  state_abrv 2025_available  year      grade  subject  value
0         AZ             No  2023  8th grade     Math   27.0
1         AZ             No  2023  8th grade  Reading   37.0
2         AZ             No  2024  8th grade     Math   28.0
3         AZ             No  2024  8th grade  Reading   35.0
4         AZ             No  2025  8th grade     Math   27.0
5         AZ             No  2025  8th grade  Reading   40.0


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




4th grade
  state_abrv 2025_available  year      grade  subject  value
0         CT            Yes  2023  4th grade     Math   65.6
1         CT            Yes  2023  4th grade  Reading   69.2
2         CT            Yes  2024  4th grade     Math   49.7
3         CT            Yes  2024  4th grade  Reading   49.4
4         CT            Yes  2025  4th grade     Math   51.4
5         CT            Yes  2025  4th grade  Reading   51.4
8th grade
  state_abrv 2025_available  year      grade  subject  value
0         CT            Yes  2023  8th grade     Math   56.1
1         CT            Yes  2023  8th grade  Reading   61.6
2         CT            Yes  2024  8th grade     Math   38.2
3         CT            Yes  2024  8th grade  Reading   49.1
4         CT            Yes  2025  8th grade     Math   40.8
5         CT            Yes  2025  8th grade  Reading   50.7


...

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




4th grade
  state_abrv 2025_available  year      grade  subject  value
0         WV            Yes  2023  4th grade     Math  45.66
1         WV            Yes  2023  4th grade  Reading  44.10
2         WV            Yes  2024  4th grade     Math  48.00
3         WV            Yes  2024  4th grade  Reading  47.00
4         WV            Yes  2025  4th grade     Math  50.00
5         WV            Yes  2025  4th grade  Reading  50.00
8th grade
  state_abrv 2025_available  year      grade  subject  value
0         WV            Yes  2023  8th grade     Math  28.21
1         WV            Yes  2023  8th grade  Reading  42.70
2         WV            Yes  2024  8th grade     Math  29.00
3         WV            Yes  2024  8th grade  Reading  41.00
4         WV            Yes  2025  8th grade     Math  33.00
5         WV            Yes  2025  8th grade  Reading  43.00


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




4th grade
  state_abrv 2025_available  year      grade  subject  value
0         WY            Yes  2023  4th grade     Math  51.10
1         WY            Yes  2023  4th grade  Reading  45.40
2         WY            Yes  2024  4th grade     Math  54.54
3         WY            Yes  2024  4th grade  Reading  49.67
4         WY            Yes  2025  4th grade     Math  57.37
5         WY            Yes  2025  4th grade  Reading  51.25
8th grade
  state_abrv 2025_available  year      grade  subject  value
0         WY            Yes  2023  8th grade     Math  49.60
1         WY            Yes  2023  8th grade  Reading  59.80
2         WY            Yes  2024  8th grade     Math  49.55
3         WY            Yes  2024  8th grade  Reading  56.86
4         WY            Yes  2025  8th grade     Math  49.77
5         WY            Yes  2025  8th grade  Reading  61.43


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




## Page 3

### NAEP

### Overall Proficiency Rates

In [12]:
#GRAPHS

g_width = 1340
g_height = 768


naep_prof = data_pull.get_naep_data_long()

us_only = naep_prof[naep_prof['state_abrv']=='US']



offset = .8
for state in state_abbreviations_priority:
    if 'US' in state:
        continue
    result = naep_prof[naep_prof['state_abrv']==state]
    # print(result.to_string())
    graphs = result.loc[:,["grade", "subject"]]
    graphs = graphs.drop_duplicates(inplace=False).sort_values(by=['grade']).reset_index(drop=True)
    for row in graphs.itertuples():
        narrowed_results = result[(result['grade']==row.grade) & (result['subject']==row.subject)].sort_values(by='year').reset_index(drop = True)
        us_vals = us_only[(us_only['grade']==row.grade) & (us_only['subject']==row.subject)].sort_values(by='year').reset_index(drop = True)
        # print(us_vals)
        # print(narrowed_results.to_string())
        
        fig = graph_naep_overall(narrowed_results,us_vals,'at_or_above_proficient', offset)
        fig.update_layout(
            font=dict(
                family='Lato',
                size=20,
                color=hunt_darkgray
            ),
            width=g_width,
            height = g_height
            )
        # graph_naep_overall()
        if row.subject == 'reading':
            if row.grade == 4:
                graph_num = 1
            elif row.grade == 8:
                graph_num = 2

        if row.subject == 'math':
            if row.grade == 4:
                graph_num = 3
            elif row.grade == 8:
                graph_num = 4


            
        if show_it:
            fig.show()
            if just_one:
                break
        if save_it:
            save_to_folder(fig,f'3.{graph_num}.png',g_width,g_height,state)
            

math from NDECoreExcel_Mathematics, Grade 4, All students_20251020161859.Xls
trouble child: nan
math from NDECoreExcel_Mathematics, Grade 8, All students_20251020161917.Xls
trouble child: nan
reading from NDECoreExcel_Reading, Grade 4, All students_20251020161841.Xls
reading from NDECoreExcel_Reading, Grade 8, All students_20251020161933.Xls


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:

## Page 4

### NAEP by region

In [13]:
#get the naep data
naep_df = data_pull.get_naep_data()
naep_df = naep_df[naep_df['state_abrv'].isnull()==False].sort_values(by='state_abrv').reset_index(drop=True)
print(naep_df.to_string())

math from NDECoreExcel_Mathematics, Grade 4, All students_20251013143833.Xls
math from NDECoreExcel_Mathematics, Grade 8, All students_20251013143853.Xls
trouble child: nan
reading from NDECoreExcel_Reading, Grade 4, All students_20251013144202.Xls
reading from NDECoreExcel_Reading, Grade 8, All students_20251013144221.Xls
    state_abrv          jurisdiction  year  grade  subject at_or_above_proficient
0           AK                Alaska  2024      4     math              29.891634
1           AK                Alaska  2024      8     math              21.549565
2           AK                Alaska  2024      8  reading               21.69763
3           AK                Alaska  2024      4  reading              21.734995
4           AL               Alabama  2024      4  reading              28.041335
5           AL               Alabama  2024      4     math              36.903023
6           AL               Alabama  2024      8  reading              21.263363
7           AL     

In [14]:
#region graphs dict setup
naep_data_dict = {}
for juri in state_abbreviations:
    if juri == 'US':
        us_data = naep_df[naep_df['state_abrv']=="US"]
        naep_us_only = dict(tuple(us_data.groupby(['grade','subject'])))
        naep_data_dict[juri] = naep_us_only 
        # continue
    # print(juri)

    else:
        region_df = get_census_regions(juri)

        region_list = region_df['state_code'].to_list()
        region_data = naep_df[naep_df['state_abrv'].isin(region_list)]
    # print(region_data.to_string())

        # Split by state
        naep_by_grade_subject_df = dict(tuple(region_data.groupby(['grade','subject'])))
        naep_data_dict[juri] = naep_by_grade_subject_df
    # Access individual DataFrames
    
    # print(gr_4_math.to_string())
    # ak_df = dfs_by_state['AK']
    # for k,v in split_dfs.items():
    #     print(k,v)


In [15]:
#graphs set up
g_width = 668
g_height = 621

##### 4.1 NAEP Region_Grade 4 Reading


In [16]:
# GRAPHS: 4th grade reading

num = 1
filename = f'4.1.png'
us_naep_proficiency = naep_data_dict.get('US')[4,'reading']
# us_naep_proficiency = us_naep_proficiency[us_naep_proficiency['state_abrv']=='US'].reset_index(drop=True)
print(us_naep_proficiency.to_string())
for juri,df in naep_data_dict.items():
    if juri == 'US':
        continue

    # print(juri)

    data_df = df[4,'reading'].sort_values(by='at_or_above_proficient', ascending=False)

    all_data = pd.concat([data_df,us_naep_proficiency]).reset_index(drop = True)
    all_data['color'] = all_data['state_abrv'].apply(lambda x: get_highlight_color(x, juri))
    
    print(all_data.to_string())

    fig = graph_4_regions(all_data)
    save_to_folder(fig,filename,g_width,g_height,juri)

    
    if show_it:
        fig.show()
        if just_one:
            break
    if save_it:
        save_to_folder(fig,filename,g_width,g_height,juri)


    state_abrv jurisdiction  year  grade  subject at_or_above_proficient
177         US     National  2024      4  reading               31.14508
  state_abrv jurisdiction  year  grade  subject at_or_above_proficient    color
0         KY     Kentucky  2024      4  reading               32.70926  #333740
1         MS  Mississippi  2024      4  reading              31.980365  #333740
2         TN    Tennessee  2024      4  reading              31.783519  #333740
3         AL      Alabama  2024      4  reading              28.041335  #DCA3EB
4         US     National  2024      4  reading               31.14508  #333740
['US', 'AL', 'TN', 'MS', 'KY']


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




  state_abrv jurisdiction  year  grade  subject at_or_above_proficient    color
0         WA   Washington  2024      4  reading              32.014151  #333740
1         HI       Hawaii  2024      4  reading              31.594583  #333740
2         CA   California  2024      4  reading              28.654573  #333740
3         OR       Oregon  2024      4  reading              26.605193  #333740
4         AK       Alaska  2024      4  reading              21.734995  #DCA3EB
5         US     National  2024      4  reading               31.14508  #333740
['US', 'AK', 'OR', 'CA', 'HI', 'WA']


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




  state_abrv jurisdiction  year  grade  subject at_or_above_proficient    color
0         UT         Utah  2024      4  reading              36.322119  #333740
1         WY      Wyoming  2024      4  reading              35.847081  #333740
2         CO     Colorado  2024      4  reading              35.655028  #333740
3         MT      Montana  2024      4  reading              31.876067  #333740
4         ID        Idaho  2024      4  reading              31.871251  #333740
5         NV       Nevada  2024      4  reading              30.026748  #333740
6         AZ      Arizona  2024      4  reading               26.44514  #DCA3EB
7         NM   New Mexico  2024      4  reading              20.293432  #333740
8         US     National  2024      4  reading               31.14508  #333740
['US', 'NM', 'AZ', 'NV', 'ID', 'MT', 'CO', 'WY', 'UT']


...

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




  state_abrv jurisdiction  year  grade  subject at_or_above_proficient    color
0         UT         Utah  2024      4  reading              36.322119  #333740
1         WY      Wyoming  2024      4  reading              35.847081  #DCA3EB
2         CO     Colorado  2024      4  reading              35.655028  #333740
3         MT      Montana  2024      4  reading              31.876067  #333740
4         ID        Idaho  2024      4  reading              31.871251  #333740
5         NV       Nevada  2024      4  reading              30.026748  #333740
6         AZ      Arizona  2024      4  reading               26.44514  #333740
7         NM   New Mexico  2024      4  reading              20.293432  #333740
8         US     National  2024      4  reading               31.14508  #333740
['US', 'NM', 'AZ', 'NV', 'ID', 'MT', 'CO', 'WY', 'UT']


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




  state_abrv          jurisdiction  year  grade  subject at_or_above_proficient    color
0         MD              Maryland  2024      4  reading              33.552784  #333740
1         FL               Florida  2024      4  reading              32.987005  #333740
2         SC        South Carolina  2024      4  reading              32.474725  #333740
3         VA              Virginia  2024      4  reading               30.72257  #333740
4         GA               Georgia  2024      4  reading              30.439746  #333740
5         NC        North Carolina  2024      4  reading              30.083994  #333740
6         DC  District of Columbia  2024      4  reading              29.635301  #DCA3EB
7         DE              Delaware  2024      4  reading              26.482094  #333740
8         WV         West Virginia  2024      4  reading              24.627309  #333740
9         US              National  2024      4  reading               31.14508  #333740
['US', 'WV', 'DE', 'D

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




##### 4.2 NAEP Region_Grade 4 Math 


In [17]:
# GRAPHS:  4th grade math
filename = f'4.2.png'
num = 1
us_naep_proficiency = naep_data_dict.get('US')[4,'math']
# us_naep_proficiency = us_naep_proficiency[us_naep_proficiency['state_abrv']=='US'].reset_index(drop=True)
print(us_naep_proficiency.to_string())
for juri,df in naep_data_dict.items():
    if juri == 'US':
        continue

    # print(juri)

    data_df = df[4,'math'].sort_values(by='at_or_above_proficient', ascending=False)

    all_data = pd.concat([data_df,us_naep_proficiency]).reset_index(drop = True)
    all_data['color'] = all_data['state_abrv'].apply(lambda x: get_highlight_color(x, juri))
    
    # print(all_data.to_string())

    fig = graph_4_regions(all_data)
    if show_it:
        fig.show()
        if just_one:
            break
    if save_it:
        save_to_folder(fig,filename,g_width,g_height,juri)
        

    state_abrv jurisdiction  year  grade subject at_or_above_proficient
179         US     National  2024      4    math              39.482018
['US', 'AL', 'MS', 'KY', 'TN']


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




['US', 'AK', 'OR', 'CA', 'WA', 'HI']
['US', 'NM', 'AZ', 'NV', 'MT', 'ID', 'CO', 'UT', 'WY']


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




['US', 'AR', 'OK', 'LA', 'TX']
['US', 'AK', 'OR', 'CA', 'WA', 'HI']


...

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




['US', 'MI', 'IL', 'WI', 'OH', 'IN']
['US', 'NM', 'AZ', 'NV', 'MT', 'ID', 'CO', 'UT', 'WY']


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




['US', 'WV', 'DC', 'DE', 'MD', 'GA', 'SC', 'VA', 'NC', 'FL']


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




##### 4.3 NAEP Region_Grade 8 Reading


In [18]:
#  GRAPHS: 8th grade reading
filename = f'4.3.png'
num = 1
us_naep_proficiency = naep_data_dict.get('US')[8,'reading']
# us_naep_proficiency = us_naep_proficiency[us_naep_proficiency['state_abrv']=='US'].reset_index(drop=True)
print(us_naep_proficiency.to_string())
for juri,df in naep_data_dict.items():
    if juri == 'US':
        continue

    # print(juri)

    data_df = df[8,'reading'].sort_values(by='at_or_above_proficient', ascending=False)

    all_data = pd.concat([data_df,us_naep_proficiency]).reset_index(drop = True)
    all_data['color'] = all_data['state_abrv'].apply(lambda x: get_highlight_color(x, juri))
    
    # print(all_data.to_string())

    fig = graph_4_regions(all_data)
    save_to_folder(fig,filename,g_width,g_height,juri)
    if show_it:
        fig.show()
        if just_one:
            break
    if save_it:
        save_to_folder(fig,filename,g_width,g_height,juri)


    state_abrv jurisdiction  year  grade  subject at_or_above_proficient
178         US     National  2024      8  reading              29.841141
['US', 'AL', 'MS', 'KY', 'TN']


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




['US', 'AK', 'OR', 'CA', 'HI', 'WA']


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




['US', 'NM', 'AZ', 'NV', 'WY', 'MT', 'UT', 'ID', 'CO']


...

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




['US', 'NM', 'AZ', 'NV', 'WY', 'MT', 'UT', 'ID', 'CO']


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




['US', 'WV', 'DE', 'DC', 'FL', 'SC', 'NC', 'VA', 'GA', 'MD']


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




##### 4.4 NAEP Region_Grade 8 Math


In [19]:
#  GRAPHS: 8th grade math
filename = f'4.4.png'
num = 1
us_naep_proficiency = naep_data_dict.get('US')[8,'math']
# us_naep_proficiency = us_naep_proficiency[us_naep_proficiency['state_abrv']=='US'].reset_index(drop=True)
print(us_naep_proficiency.to_string())
for juri,df in naep_data_dict.items():
    if juri == 'US':
        continue

    # print(juri)

    data_df = df[8,'math'].sort_values(by='at_or_above_proficient', ascending=False)

    all_data = pd.concat([data_df,us_naep_proficiency]).reset_index(drop = True)
    all_data['color'] = all_data['state_abrv'].apply(lambda x: get_highlight_color(x, juri))
    
    print(all_data.to_string())

    fig = graph_4_regions(all_data)

    if show_it:
        fig.show()
        if just_one:
            break
    if save_it:
        save_to_folder(fig,filename,g_width,g_height,juri)



    state_abrv jurisdiction  year  grade subject at_or_above_proficient
176         US     National  2024      8    math              28.029949
  state_abrv jurisdiction  year  grade subject at_or_above_proficient    color
0         TN    Tennessee  2024      8    math              30.514863  #333740
1         KY     Kentucky  2024      8    math              24.323528  #333740
2         MS  Mississippi  2024      8    math              21.582686  #333740
3         AL      Alabama  2024      8    math              18.367256  #DCA3EB
4         US     National  2024      8    math              28.029949  #333740
['US', 'AL', 'MS', 'KY', 'TN']


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




  state_abrv jurisdiction  year  grade subject at_or_above_proficient    color
0         WA   Washington  2024      8    math              29.505684  #333740
1         CA   California  2024      8    math              25.252664  #333740
2         OR       Oregon  2024      8    math              23.746205  #333740
3         HI       Hawaii  2024      8    math              23.150019  #333740
4         AK       Alaska  2024      8    math              21.549565  #DCA3EB
5         US     National  2024      8    math              28.029949  #333740
['US', 'AK', 'HI', 'OR', 'CA', 'WA']
  state_abrv jurisdiction  year  grade subject at_or_above_proficient    color
0         UT         Utah  2024      8    math              35.061626  #333740
1         CO     Colorado  2024      8    math               32.47483  #333740
2         MT      Montana  2024      8    math              32.097171  #333740
3         ID        Idaho  2024      8    math              30.915762  #333740
4         WY   

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




  state_abrv jurisdiction  year  grade subject at_or_above_proficient    color
0         TX        Texas  2024      8    math               23.68881  #333740
1         LA    Louisiana  2024      8    math              20.842909  #333740
2         AR     Arkansas  2024      8    math              19.817914  #DCA3EB
3         OK     Oklahoma  2024      8    math              17.371604  #333740
4         US     National  2024      8    math              28.029949  #333740
['US', 'OK', 'AR', 'LA', 'TX']
  state_abrv jurisdiction  year  grade subject at_or_above_proficient    color
0         WA   Washington  2024      8    math              29.505684  #333740
1         CA   California  2024      8    math              25.252664  #DCA3EB
2         OR       Oregon  2024      8    math              23.746205  #333740
3         HI       Hawaii  2024      8    math              23.150019  #333740
4         AK       Alaska  2024      8    math              21.549565  #333740
5         US     Nati

...

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




  state_abrv          jurisdiction  year  grade subject at_or_above_proficient    color
0         NC        North Carolina  2024      8    math              31.465277  #333740
1         VA              Virginia  2024      8    math              28.512894  #333740
2         MD              Maryland  2024      8    math              24.728188  #333740
3         SC        South Carolina  2024      8    math              23.737961  #333740
4         GA               Georgia  2024      8    math              23.657747  #333740
5         FL               Florida  2024      8    math              21.116375  #333740
6         DC  District of Columbia  2024      8    math              20.088919  #333740
7         DE              Delaware  2024      8    math              19.339533  #333740
8         WV         West Virginia  2024      8    math              17.754009  #DCA3EB
9         US              National  2024      8    math              28.029949  #333740
['US', 'WV', 'DE', 'DC', 'FL', '

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




  state_abrv jurisdiction  year  grade subject at_or_above_proficient    color
0         UT         Utah  2024      8    math              35.061626  #333740
1         CO     Colorado  2024      8    math               32.47483  #333740
2         MT      Montana  2024      8    math              32.097171  #333740
3         ID        Idaho  2024      8    math              30.915762  #333740
4         WY      Wyoming  2024      8    math              30.044219  #DCA3EB
5         AZ      Arizona  2024      8    math              25.794746  #333740
6         NV       Nevada  2024      8    math              20.199962  #333740
7         NM   New Mexico  2024      8    math              13.942698  #333740
8         US     National  2024      8    math              28.029949  #333740
['US', 'NM', 'NV', 'AZ', 'WY', 'ID', 'MT', 'CO', 'UT']
  state_abrv          jurisdiction  year  grade subject at_or_above_proficient    color
0         NC        North Carolina  2024      8    math            

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




### State Assessment rigor


In [20]:
# data set up
state_cut_df = data_pull.get_naep_workbook_data(option='state cut')
state_cut_df = state_cut_df.iloc[4:,:].reset_index(drop=True)
state_cut_df.columns = state_cut_df.iloc[0,:]
state_cut_df = state_cut_df.iloc[1:,:].reset_index(drop=True)


# Rename first column to 'state'
state_cut_df.columns = ['state'] + list(state_cut_df.columns[1:])

# Melt the dataframe
result = []

for col in state_cut_df.columns[1:]:
    # Extract subject and year from column name
    parts = col.split()
    subject = parts[0]
    year = parts[1]
    
    # Create a temporary dataframe
    temp = state_cut_df[['state', col]].copy()
    temp.columns = ['state', 'value']
    temp['year'] = year
    temp['subject'] = subject
    
    result.append(temp)

# Concatenate all dataframes
final_df = pd.concat(result, ignore_index=True)

# Reorder columns
assess_rigor = final_df[['state', 'year', 'subject', 'value']]

# Replace em dashes and en dashes with NaN
assess_rigor['value'] = assess_rigor['value'].replace(['—', '–'], pd.NA)

# Convert value to numeric
assess_rigor['value'] = pd.to_numeric(assess_rigor['value'], errors='coerce')
assess_rigor['state_abrv'] = assess_rigor['state'].apply(get_state_abrv_from_lower)
popped = assess_rigor.pop('state_abrv')
assess_rigor.insert(0,'state_abrv', popped)

# print(assess_rigor.to_string())


In [21]:
# GRAPHS 
g_width = 1340
g_height = 575
offset = 10
grade = 4
subjects = {'reading':'4.5.png','math':'4.6.png'}

for state in state_abbreviations_priority:
    if state == 'US':
        continue
    for sub,filename in subjects.items():
        # filename_plus = f'{sub}{filename}'
        result = assess_rigor[(assess_rigor['state_abrv']==state) & (assess_rigor['subject']==subject)].reset_index(drop=True).sort_values(by=['state_abrv','year'])
        print(bordered(state))
        print(result.to_string())
        ymin = min(result['value'])-50

        fig = graph_state_cut(result, 'value', state, result['year'])
        fig.update_yaxes(
            showticklabels=False,  # Hide tick labels
            showgrid=False,        # Hide gridlines
            zeroline=False         # Hide zero line
        )
        fig.update_layout(
            xaxis=dict(
                showgrid=False, 
                showline=True, 
                linecolor=hunt_darkgray, 
                tickmode='array', 
                tickvals=result['year']
            ),
            yaxis=dict(
                range=[ymin, 275],
                showgrid=False, 
                showline=True, 
                linecolor=hunt_darkgray
            ),
            font=dict(
                family='Lato',
                size=20,
                color=hunt_darkgray
            ),
            width = g_width,
            height= g_height)
        
        text_y_positions = [float(x)-offset for x in result['value']]
        # Add text labels
        fig.add_trace(go.Scatter(
            x=result['year'].to_list(),
            y=text_y_positions,  # Use calculated positions
            mode='text',
            showlegend=False,
            text=result['value'].apply(lambda x: str(int(round(x))) if pd.notna(x) else ''),
            textfont=dict(
                    size=20,           # Font size
                    color=hunt_darkgray,     # Font color
                    family='Lato'     # Font family
                )
        ))
        save_to_folder(fig,filename,g_width,g_height,state)
        # fig.show()
        # break


┌──┐
│AZ│
└──┘
  state_abrv    state  year subject       value
5         AZ  Arizona  2011    math  226.422391
4         AZ  Arizona  2013    math  228.887179
3         AZ  Arizona  2015    math  246.199139
2         AZ  Arizona  2017    math  240.060051
1         AZ  Arizona  2019    math  241.744860
0         AZ  Arizona  2022    math  240.000000


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




┌──┐
│AZ│
└──┘
  state_abrv    state  year subject       value
5         AZ  Arizona  2011    math  226.422391
4         AZ  Arizona  2013    math  228.887179
3         AZ  Arizona  2015    math  246.199139
2         AZ  Arizona  2017    math  240.060051
1         AZ  Arizona  2019    math  241.744860
0         AZ  Arizona  2022    math  240.000000
┌──┐
│CT│
└──┘
  state_abrv        state  year subject       value
5         CT  Connecticut  2011    math  211.811130
4         CT  Connecticut  2013    math  214.060903
3         CT  Connecticut  2015    math  245.618408
2         CT  Connecticut  2017    math  241.186372
1         CT  Connecticut  2019    math  241.860524
0         CT  Connecticut  2022    math  240.000000


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




┌──┐
│CT│
└──┘
  state_abrv        state  year subject       value
5         CT  Connecticut  2011    math  211.811130
4         CT  Connecticut  2013    math  214.060903
3         CT  Connecticut  2015    math  245.618408
2         CT  Connecticut  2017    math  241.186372
1         CT  Connecticut  2019    math  241.860524
0         CT  Connecticut  2022    math  240.000000
┌──┐
│DC│
└──┘
  state_abrv                 state  year subject       value
5         DC  District of Columbia  2011    math  225.292646
4         DC  District of Columbia  2013    math  221.402174
3         DC  District of Columbia  2015    math  251.320836
2         DC  District of Columbia  2017    math  247.901807
1         DC  District of Columbia  2019    math  244.177242
0         DC  District of Columbia  2022    math  252.000000


...

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




┌──┐
│WY│
└──┘
  state_abrv    state  year subject       value
5         WY  Wyoming  2011    math  221.723355
4         WY  Wyoming  2013    math  225.206397
3         WY  Wyoming  2015    math  247.495713
2         WY  Wyoming  2017    math  243.903275
1         WY  Wyoming  2019    math  245.974176
0         WY  Wyoming  2022    math  242.000000


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




┌──┐
│WY│
└──┘
  state_abrv    state  year subject       value
5         WY  Wyoming  2011    math  221.723355
4         WY  Wyoming  2013    math  225.206397
3         WY  Wyoming  2015    math  247.495713
2         WY  Wyoming  2017    math  243.903275
1         WY  Wyoming  2019    math  245.974176
0         WY  Wyoming  2022    math  242.000000


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




## Page 5

#### Proficiency %, by Race/Ethnicity


In [22]:
longitudal_data = data_pull.get_naep_workbook_data()
longitudal_data = longitudal_data.drop(2, axis=1)

longitudal_data.columns = [str(x).strip().replace('-','').replace(' ','_').lower() for x in longitudal_data.iloc[2,:]]
longitudal_data = longitudal_data.iloc[3:,:].reset_index(drop=True)


longitudal_data['state_abrv'] = longitudal_data['state'].apply(get_state_abrv_from_lower)
popped_col = longitudal_data.pop('state_abrv')
longitudal_data.insert(0,'state_abrv', popped_col)
columns = []
data_cols = []
for col in longitudal_data.columns:
    if 'proficiency' in str(col):
        grade_match = re.search(r'grade_(\d)', str(col))
        # print(grade_match)
        grade = grade_match.group(1)
        subj = col.split('_')[-1]
        col_name = f'{grade}th_{subj}_prof'
        columns.append(col_name)
        data_cols.append(col_name)
    else:
        columns.append(col)

longitudal_data.columns = columns
print(longitudal_data.to_string())



     state_abrv                 state                       subgroup  year 4th_math_prof 4th_reading_prof 8th_math_prof 8th_reading_prof
0            US              National                          White  2024            51               39            38               37
1            US              National                          Black  2024            19               17            10               16
2            US              National                       Hispanic  2024            27               21            15               19
3            US              National         Asian/Pacific Islander  2024            63               51            57               53
4            US              National  American Indian/Alaska Native  2024            19               14            11               18
5            US              National              Two or more races  2024            43               35            31               35
6            AL               Alabama    

In [23]:
#RE GRAPHS needs to be run for both subjects

g_width = 1142
g_height = 541
#math or reading
subject = 'reading'

subjects = {'reading':'5.1.png','math':'5.2.png'}

long_dat_dict = {}
for i,jur in enumerate(state_abbreviations_priority):
    if jur == "US":
        continue
    
    # if i>10:
    #     break
    for sub,filename in subjects.items():
        result = longitudal_data[(longitudal_data['state_abrv']==jur)&(longitudal_data['year']>=2011)].reset_index(drop=True)
        # print(result.to_string())
        # for year in result['year']:
        #     print(type(year))
        years = sorted(get_col_uniq_vals(result['year']))
        values = sorted(result['year'])
        # x_range = [2011,2025]
        fig = graph_5_multi_series(result,f'4th_{sub}_prof','subgroup',years)
        
        fig.update_layout(
            width=g_width,
            height = g_height, 
            font=dict(
                    family='Lato',
                    size=22,
                    color=hunt_darkgray
                )
        )
        # full_filename = f'{sub}{filename}'
        # save_to_folder(fig,filename,g_width,g_height,jur)
        if show_it:
            fig.show()
            if just_one:
                break
        if save_it:
            save_to_folder(fig,filename,g_width,g_height,jur)

        # fig.show()
    # break
    

6


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6


### Subgroup: Free/Reduced Lunch Elgibility (FRL)
table: 
NAEP Proficiency Rates by Free/Reduced Lunch | Grades 4 | 2009 - 2022

description:
The following graphs outline a breakdown of the percentage of students considered Proficient or better by Free/Reduced Lunch eligibility



In [17]:
frl_long_data = data_pull.get_naep_workbook_data(option='frl')
frl_long_data = frl_long_data.drop(3, axis=1)

frl_long_data.columns = [str(x).strip().replace('-','').replace(' ','_').lower() for x in frl_long_data.iloc[2,:]]
frl_long_data = frl_long_data.iloc[3:,:].reset_index(drop=True)


frl_long_data['state_abrv'] = frl_long_data['state'].apply(get_state_abrv_from_lower)
popped_col = frl_long_data.pop('state_abrv')
frl_long_data.insert(0,'state_abrv', popped_col)

frl_long_data.columns = ['state_abrv','year','state','eligibility','math_prof','reading_prof']
frl_long_data = frl_long_data[~frl_long_data['eligibility'].str.contains('information', case=False)]

# print(frl_long_data.to_string())


In [18]:
#FRL GRAPHS

g_width = 1142
g_height = 541
#math or reading


subjects = {'reading':'5.3.png','math':'5.4.png'}

frl_long_dat_dict = {}
for i,jur in enumerate(state_abbreviations_priority):
    if jur == "US":
        continue

    for sub, filename in subjects.items():
        result = frl_long_data[(frl_long_data['state_abrv']==jur)&(frl_long_data['year']>=2011)].reset_index(drop=True)
        # print(result.to_string())
        # for year in result['year']:
        #     print(type(year))
        years = sorted(get_col_uniq_vals(result['year']))
        values = sorted(result['year'])
        # x_range = [2011,2025]
        fig = graph_5_multi_series(result,f'{sub}_prof','eligibility',years)
        
        fig.update_layout(
            width=g_width,
            height = g_height, 
            font=dict(
                    family='Lato',
                    size=22,
                    color=hunt_darkgray
                )
        )
        if show_it:
            fig.show()
            if just_one:
                break
        if save_it:
            save_to_folder(fig,filename,g_width,g_height,jur)


2
2
2


C:\Users\clutz\AppData\Local\Temp\ipykernel_33100\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2


### Subgroup: English Language Learners (ELL)

table: 
NAEP Proficiency Rates for English Language Learners | Grades 4 | 2010 - 2024

description:
The following graphs outline a breakdown of the percentage of students considered Proficient or better English Language Learners status



In [19]:
ell_long_data = data_pull.get_naep_workbook_data(option='ell')
# ell_long_data = ell_long_data.drop(3, axis=1)

ell_long_data.columns = [str(x).strip().replace('-','').replace(' ','_').replace('/jurisdiction','').lower() for x in ell_long_data.iloc[2,:]]
ell_long_data = ell_long_data.iloc[3:,:].reset_index(drop=True)


ell_long_data['state_abrv'] = ell_long_data['state/jurisdiction'].apply(get_state_abrv_from_lower)
popped_col = ell_long_data.pop('state_abrv')
ell_long_data.insert(0,'state_abrv', popped_col)

ell_long_data.columns = ['state_abrv','year','state','ell_status','math_prof','reading_prof']
ell_long_data = ell_long_data[~ell_long_data['ell_status'].str.contains('information', case=False)]
print(ell_long_data.to_string())



    state_abrv  year                 state ell_status math_prof reading_prof
0           US  2024              National        ELL        16            8
1           US  2024              National    Not ELL        43           35
2           AL  2024               Alabama        ELL         9            3
3           AL  2024               Alabama    Not ELL        39           30
4           AK  2024                Alaska        ELL         9            6
5           AK  2024                Alaska    Not ELL        33           24
6           AZ  2024               Arizona        ELL         5            1
7           AZ  2024               Arizona    Not ELL        37           30
8           AR  2024              Arkansas        ELL         8            4
9           AR  2024              Arkansas    Not ELL        34           31
10          CA  2024            California        ELL        12            4
11          CA  2024            California    Not ELL        42           36

In [20]:
# ELL GRAPHS
#math or reading
# subject="math"
# filename = f'5.4.png'
g_width = 1142
g_height = 541
subjects = {'reading':'5.5.png','math':'5.6.png'}

ell_long_dat_dict = {}
for i, jur in enumerate(state_abbreviations_priority):
    if jur == "US":
        continue

    for sub, filename in subjects.items():
        result = ell_long_data[(ell_long_data['state_abrv']==jur) & (ell_long_data['year']>=2011)].reset_index(drop=True)
        years = sorted(get_col_uniq_vals(result['year']))
        
        fig = graph_5_multi_series(result, f'{sub}_prof', 'ell_status', years, label_all=True)
        
        fig.update_layout(
            width=g_width,
            height=g_height,
            font=dict(
                family='Lato',
                size=22,
                color=hunt_darkgray
            )
        )
        
        # Override yaxis separately to ensure it takes effect
        fig.update_yaxes(
            range=[-10, 80],
            tickvals=list(range(0, 90, 10)),
            ticksuffix='%',
            showgrid=False,
            showline=True,
            linecolor='black'
        )
        if show_it:
            fig.show()
            if just_one:
                break
        if save_it:
            save_to_folder(fig,filename,g_width,g_height,jur)


    # break
    

2


C:\Users\clutz\AppData\Local\Temp\ipykernel_33100\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2


### Subgroup: Disability Status (SWD)

table: 
NAEP Proficiency Rates for Students with Disabilities | Grades 4 | 2010 - 2024

description:
The following graphs outline a breakdown of the percentage of students considered Proficient or better by disability status


In [21]:
swd_long_data = data_pull.get_naep_workbook_data(option='swd')
# swd_long_data = swd_long_data.drop(3, axis=1)

swd_long_data.columns = [str(x).strip().replace('-','').replace(' ','_').replace('/jurisdiction','').lower() for x in swd_long_data.iloc[2,:]]
swd_long_data = swd_long_data.iloc[3:,:].reset_index(drop=True)


swd_long_data['state_abrv'] = swd_long_data['state/jurisdiction'].apply(get_state_abrv_from_lower)
popped_col = swd_long_data.pop('state_abrv')
swd_long_data.insert(0,'state_abrv', popped_col)

swd_long_data.columns = ['state_abrv','year','state','ell_status','math_prof','reading_prof']
swd_long_data = swd_long_data[~swd_long_data['ell_status'].str.contains('information', case=False)]
print(swd_long_data.to_string())



    state_abrv  year                 state                                    ell_status math_prof reading_prof
0           US  2024              National      Identified as students with disabilities        16           10
1           US  2024              National  Not identified as students with disabilities        44           35
2           AL  2024               Alabama      Identified as students with disabilities        17            6
3           AL  2024               Alabama  Not identified as students with disabilities        41           32
4           AK  2024                Alaska      Identified as students with disabilities        10            6
5           AK  2024                Alaska  Not identified as students with disabilities        34           25
6           AZ  2024               Arizona      Identified as students with disabilities        13            8
7           AZ  2024               Arizona  Not identified as students with disabilities        37      

In [22]:
#SWD GRAPHS
g_width = 1142
g_height = 541
#math or reading

subjects = {'reading':'5.7.png','math':'5.8.png'}

swd_long_dat_dict = {}
for i,jur in enumerate(state_abbreviations_priority):
    if jur == "US":
        continue

    for sub,filename in subjects.items():
        
        result = swd_long_data[(swd_long_data['state_abrv']==jur)&(swd_long_data['year']>=2011)].reset_index(drop=True)
        # print(result.to_string())
        # for year in result['year']:
        #     print(type(year))
        years = sorted(get_col_uniq_vals(result['year']))
        values = sorted(result['year'])
        # x_range = [2011,2025]
        fig = graph_5_multi_series(result,f'{sub}_prof','ell_status',years, label_all=True)
        fig.update_layout(
                width=g_width,
                height=g_height,
                font=dict(
                    family='Lato',
                    size=22,
                    color=hunt_darkgray
                )
            )
            
        # Override yaxis separately to ensure it takes effect
        fig.update_yaxes(
            range=[-10, 80],
            tickvals=list(range(0, 90, 10)),
            ticksuffix='%',
            showgrid=False,
            showline=True,
            linecolor='black'
        )
        if show_it:
            fig.show()
            if just_one:
                break
        if save_it:
            save_to_folder(fig,filename,g_width,g_height,jur)


2
2


C:\Users\clutz\AppData\Local\Temp\ipykernel_33100\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2


## Page 6

### chronic absenteesism (Race)

In [30]:
# set up
chron_abs_df = data_pull.get_collected_data(metric='chron_abs_race', no_header = True)
chron_abs_df = chron_abs_df.iloc[1:, 0:11]
chron_abs_df = chron_abs_df.drop(columns=chron_abs_df.columns[1:4])

# print(chron_abs_df.to_string())

# Set proper column names
chron_abs_df.columns = [
    'state_abrv', 'year', 'White', 'Black', 'Hispanic', 'Asian',
    'American Indian/Alaska Native', 'Native Hawaiian/Other Pacific Islander'
]

# Clean year column to pick later year if range
def extract_later_year(val):
    if pd.isna(val):
        return np.nan
    years = re.findall(r'\d{4}', str(val))
    return int(years[-1]) if years else np.nan

chron_abs_df['year'] = chron_abs_df['year'].apply(extract_later_year)

# Melt into long format
df_long = chron_abs_df.melt(
    id_vars=['state_abrv', 'year'],
    var_name='group',
    value_name='value'
)

# Normalize values to percentages
def normalize_percentage(x):
    try:
        x = float(str(x).replace('%','').replace('%%',''))  # remove stray % symbols
        if x > 1.5:  # likely already in percent form
            return x
        else:        # decimal → percent
            return x * 100
    except:
        return np.nan

df_long['value'] = df_long['value'].apply(normalize_percentage)

# Optional: sort for readability
chron_abs_df = df_long.sort_values(['state_abrv', 'group']).reset_index(drop=True)

# Preview
print(chron_abs_df.to_string())


    state_abrv    year                                   group  value
0           AK  2022.0           American Indian/Alaska Native  57.60
1           AK  2022.0                                   Asian  43.60
2           AK  2022.0                                   Black  34.90
3           AK  2022.0                                Hispanic  43.30
4           AK  2022.0  Native Hawaiian/Other Pacific Islander    NaN
5           AK  2022.0                                   White  37.80
6           AL  2024.0           American Indian/Alaska Native  16.00
7           AL  2024.0                                   Asian   5.10
8           AL  2024.0                                   Black  18.40
9           AL  2024.0                                Hispanic  11.90
10          AL  2024.0  Native Hawaiian/Other Pacific Islander  13.60
11          AL  2024.0                                   White  13.50
12          AR  2022.0           American Indian/Alaska Native   0.60
13          AR  2022

### chronic absenteesism (Other Subgroup)

In [31]:
# GRAPHS chronic absenteeism
g_width = 2703
g_height = 628

offset = 1
filename = f'6.1.png'
for state in state_abbreviations_priority:
    if state == 'US':
        continue
    result = chron_abs_df[chron_abs_df['state_abrv']==state].reset_index(drop=True)
    # print(result.to_string())
    fig = graph_chron_abs(result, 'value')
    fig.update_layout(
        width=g_width,
        height=g_height,
        font=dict(
            family='Lato',
            size=22,
            color=hunt_darkgray
        )
    )
    fig.update_yaxes(
        visible=False
    )
    if show_it:
        fig.show()
        if just_one:
            break
    if save_it:
        save_to_folder(fig,filename,g_width,g_height,state)

    


CATEGORIES
['American Indian/Alaska Native', 'Asian/Pacific Islander', 'Native Hawaiian/Other Pacific Islander', 'Asian', 'Hispanic', 'Black', 'White', 'Two or More']
['#5DC2D0', '#63007E', '#280033', '#63007E', '#D77900', '#002C99', '#89B8EA', '#00B188']
                                    group  value               group_key  \
0           American Indian/Alaska Native   39.8            nat_am_or_ak   
1                  Asian/Pacific Islander    NaN  asian_pacific_islander   
2  Native Hawaiian/Other Pacific Islander   28.2                 pacific   
3                                   Asian    9.3                   asian   
4                                Hispanic   28.4                hispanic   
5                                   Black   24.5                   black   
6                                   White   18.5                   white   
7                             Two or More    NaN             two_or_more   

   order  
0      1  
1      2  
2      2  
3      2  
4   

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




CATEGORIES
['American Indian/Alaska Native', 'Asian/Pacific Islander', 'Native Hawaiian/Other Pacific Islander', 'Asian', 'Hispanic', 'Black', 'White', 'Two or More']
['#5DC2D0', '#63007E', '#280033', '#63007E', '#D77900', '#002C99', '#89B8EA', '#00B188']
                                    group  value               group_key  \
0           American Indian/Alaska Native   22.3            nat_am_or_ak   
1                  Asian/Pacific Islander    NaN  asian_pacific_islander   
2  Native Hawaiian/Other Pacific Islander   16.4                 pacific   
3                                   Asian   11.0                   asian   
4                                Hispanic   24.9                hispanic   
5                                   Black   23.1                   black   
6                                   White   10.8                   white   
7                             Two or More    NaN             two_or_more   

   order  
0      1  
1      2  
2      2  
3      2  
4   

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




CATEGORIES
['American Indian/Alaska Native', 'Asian/Pacific Islander', 'Native Hawaiian/Other Pacific Islander', 'Asian', 'Hispanic', 'Black', 'White', 'Two or More']
['#5DC2D0', '#63007E', '#280033', '#63007E', '#D77900', '#002C99', '#89B8EA', '#00B188']
                                    group  value               group_key  \
0           American Indian/Alaska Native   39.5            nat_am_or_ak   
1                  Asian/Pacific Islander    NaN  asian_pacific_islander   
2  Native Hawaiian/Other Pacific Islander   26.1                 pacific   
3                                   Asian   13.8                   asian   
4                                Hispanic   34.6                hispanic   
5                                   Black   49.1                   black   
6                                   White    9.3                   white   
7                             Two or More    NaN             two_or_more   

   order  
0      1  
1      2  
2      2  
3      2  
4   

...

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




CATEGORIES
['American Indian/Alaska Native', 'Asian/Pacific Islander', 'Native Hawaiian/Other Pacific Islander', 'Asian', 'Hispanic', 'Black', 'White', 'Two or More']
['#5DC2D0', '#63007E', '#280033', '#63007E', '#D77900', '#002C99', '#89B8EA', '#00B188']
                                    group  value               group_key  \
0           American Indian/Alaska Native   45.5            nat_am_or_ak   
1                  Asian/Pacific Islander    NaN  asian_pacific_islander   
2  Native Hawaiian/Other Pacific Islander   49.4                 pacific   
3                                   Asian   16.4                   asian   
4                                Hispanic   33.6                hispanic   
5                                   Black   29.6                   black   
6                                   White   24.4                   white   
7                             Two or More    NaN             two_or_more   

   order  
0      1  
1      2  
2      2  
3      2  
4   

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




CATEGORIES
['American Indian/Alaska Native', 'Asian/Pacific Islander', 'Native Hawaiian/Other Pacific Islander', 'Asian', 'Hispanic', 'Black', 'White', 'Two or More']
['#5DC2D0', '#63007E', '#280033', '#63007E', '#D77900', '#002C99', '#89B8EA', '#00B188']
                                    group  value               group_key  \
0           American Indian/Alaska Native    6.3            nat_am_or_ak   
1                  Asian/Pacific Islander    NaN  asian_pacific_islander   
2  Native Hawaiian/Other Pacific Islander    0.2                 pacific   
3                                   Asian    0.4                   asian   
4                                Hispanic   17.9                hispanic   
5                                   Black    1.0                   black   
6                                   White   69.9                   white   
7                             Two or More    NaN             two_or_more   

   order  
0      1  
1      2  
2      2  
3      2  
4   

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




In [32]:
# chronic absenteesism other
chron_abs_df_other = data_pull.get_collected_data(metric='chron_abs_other', no_header = True)
chron_abs_df_other = chron_abs_df_other.drop(columns=chron_abs_df_other.columns[1:6])

chron_abs_df_other.columns = ['state', 'year']+list(chron_abs_df_other.iloc[1,2:])
chron_abs_df_other = chron_abs_df_other.iloc[2:, :].reset_index(drop=True)
chron_abs_df_other = chron_abs_df_other.drop(columns=['notes', 'date pulled'])
print(chron_abs_df_other.columns)
# print(chron_abs_df_other.to_string())


Index(['state', 'year', 'All', 'Economically Disadvantaged',
       'English Language Learners', 'Students With Disabilities'],
      dtype='object')


In [ ]:
# GRAPHS chronic absenteeism other
g_width = 1355
g_height = 588
filename = '6.2.png'
for state in state_abbreviations_priority:
    result = chron_abs_df_other[chron_abs_df_other['state']==state].reset_index()
    print(result.to_string())
    fig = graph_other_chron_abs(result)
    fig.update_layout(
        width=g_width,
        height=g_height,
        font=dict(
            family='Lato',
            size=22,
            color=hunt_darkgray
        )
    )
    fig.update_yaxes(
        visible=False
    )
    if show_it:
        fig.show()
        if just_one:
            break
    if save_it:
        save_to_folder(fig,filename,g_width,g_height,state)

   index state       year    All Economically Disadvantaged English Language Learners Students With Disabilities
0      2    AZ  2023-2024  0.244                      0.306                     0.276                      0.295
['All', 'Economically Disadvantaged', 'English Language Learners', 'Students With Disabilities']
   index state       year    All Economically Disadvantaged English Language Learners Students With Disabilities
0      6    CT  2024-2025  0.172                      0.282                     0.238                      0.267

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).





['All', 'Economically Disadvantaged', 'English Language Learners', 'Students With Disabilities']
   index state       year    All Economically Disadvantaged English Language Learners Students With Disabilities
0      8    DC  2023-2024  0.399                      0.558                     0.307                      0.485
['All', 'Economically Disadvantaged', 'English Language Learners', 'Students With Disabilities']


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




   index state       year    All Economically Disadvantaged English Language Learners Students With Disabilities
0      7    DE  2022-2023  0.243                      0.114                     0.029                      0.062
['All', 'Economically Disadvantaged', 'English Language Learners', 'Students With Disabilities']
   index state       year    All Economically Disadvantaged English Language Learners Students With Disabilities
0     10    GA  2023-2024  0.218                      0.258                       0.2                      0.268
['All', 'Economically Disadvantaged', 'English Language Learners', 'Students With Disabilities']


...

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




   index state       year    All Economically Disadvantaged English Language Learners Students With Disabilities
0     47    WA  2023-2024  0.273                      0.356                     0.326                      0.347
['All', 'Economically Disadvantaged', 'English Language Learners', 'Students With Disabilities']
   index state  year     All Economically Disadvantaged English Language Learners Students With Disabilities
0     48    WV  2025  0.2264                     0.2923                    0.1982                     0.2787
['All', 'Economically Disadvantaged', 'English Language Learners', 'Students With Disabilities']


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




   index state       year    All Economically Disadvantaged English Language Learners Students With Disabilities
0     50    WY  2022-2023  0.341                      0.171                     0.012                      0.068
['All', 'Economically Disadvantaged', 'English Language Learners', 'Students With Disabilities']


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




In [34]:
#out of school suspension
oos_df = data_pull.get_collected_data(metric='suspension_race', no_header = True)
oos_df.columns = list(oos_df.iloc[1,:])
oos_df = oos_df.iloc[2:,:].reset_index(drop=True)


oos_df = oos_df.drop(columns=oos_df.columns[[1,9,10,11]])
print(oos_df.columns)


Index(['state', 'White', 'Black', 'Hispanic', 'Asian',
       'American Indian or Alaska Native',
       'Native Hawaiian or Other Pacific Islander', 'Two or More Races'],
      dtype='object')


In [35]:
# GRAPHS out of school suspension
g_width = 2828
g_height = 648
filename = '6.3.png'

us_only = oos_df[oos_df['state']=="US"].reset_index(drop=True)

for state in state_abbreviations_priority:
    result = oos_df[oos_df['state']==state].reset_index(drop = True)
    print(result.to_string())
    print(us_only.to_string())
    fig = graph_oos(result, us_only, state)
    fig.update_layout(
        width=g_width,
        height=g_height,
        font=dict(
            family='Lato',
            size=22,
            color=hunt_darkgray
        )
    )
    fig.update_yaxes(
        visible=False
    )
    if show_it:
        fig.show()
        if just_one:
            break
    if save_it:
        save_to_folder(fig,filename,g_width,g_height,state)


# oos_df.columns = ['state', 'year']+list(oos_df.iloc[1,2:])
# oos_df = oos_df.iloc[2:, :].reset_index(drop=True)
# oos_df = oos_df.drop(columns=['notes', 'date pulled'])
# print(oos_df.columns)


  state White  Black Hispanic Asian American Indian or Alaska Native Native Hawaiian or Other Pacific Islander Two or More Races
0    AZ   0.4  0.117    0.375   <1%                            0.027                                       <1%             0.068
  state  White  Black Hispanic Asian American Indian or Alaska Native Native Hawaiian or Other Pacific Islander Two or More Races
0    US  0.461  0.304    0.153   <1%                            0.015                                       <1%             0.058
['White', 'Black', 'Hispanic', 'Asian', 'American Indian or Alaska Native', 'Native Hawaiian or Other Pacific Islander', 'Two or More Races']


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




  state White  Black Hispanic  Asian American Indian or Alaska Native Native Hawaiian or Other Pacific Islander Two or More Races
0    CT  0.37  0.249    0.287  0.024                                0                                         0             0.069
  state  White  Black Hispanic Asian American Indian or Alaska Native Native Hawaiian or Other Pacific Islander Two or More Races
0    US  0.461  0.304    0.153   <1%                            0.015                                       <1%             0.058
['White', 'Black', 'Hispanic', 'Asian', 'American Indian or Alaska Native', 'Native Hawaiian or Other Pacific Islander', 'Two or More Races']


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




  state White Black Hispanic Asian American Indian or Alaska Native Native Hawaiian or Other Pacific Islander Two or More Races
0    DC     0     0        0     0                                0                                         0                 0
  state  White  Black Hispanic Asian American Indian or Alaska Native Native Hawaiian or Other Pacific Islander Two or More Races
0    US  0.461  0.304    0.153   <1%                            0.015                                       <1%             0.058
['White', 'Black', 'Hispanic', 'Asian', 'American Indian or Alaska Native', 'Native Hawaiian or Other Pacific Islander', 'Two or More Races']


...

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




  state  White  Black Hispanic  Asian American Indian or Alaska Native Native Hawaiian or Other Pacific Islander Two or More Races
0    WV  0.846  0.093    0.011  <1.0%                                0                                         0             0.048
  state  White  Black Hispanic Asian American Indian or Alaska Native Native Hawaiian or Other Pacific Islander Two or More Races
0    US  0.461  0.304    0.153   <1%                            0.015                                       <1%             0.058
['White', 'Black', 'Hispanic', 'Asian', 'American Indian or Alaska Native', 'Native Hawaiian or Other Pacific Islander', 'Two or More Races']


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




  state  White  Black Hispanic Asian American Indian or Alaska Native Native Hawaiian or Other Pacific Islander Two or More Races
0    WY  0.668  0.028    0.227     0                            0.031                                         0             0.046
  state  White  Black Hispanic Asian American Indian or Alaska Native Native Hawaiian or Other Pacific Islander Two or More Races
0    US  0.461  0.304    0.153   <1%                            0.015                                       <1%             0.058
['White', 'Black', 'Hispanic', 'Asian', 'American Indian or Alaska Native', 'Native Hawaiian or Other Pacific Islander', 'Two or More Races']


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




## Page 7

### Public HS Graduation Rate

In [36]:
grad_rate_df = data_pull.get_collected_data(metric='hs_grad_rate')
grad_rate_df.columns = grad_rate_df.iloc[0,:].reset_index(drop=True)
grad_rate_df = grad_rate_df.iloc[2:,:].reset_index(drop=True)
grad_rate_df = grad_rate_df.loc[:,['state', '2022-2023 data', '2023-2024 HS grad rate']]
grad_rate_df.columns = ['state_abrv', '2023', '2024']
grad_rate_df = grad_rate_df.dropna(how='all')


# Melt the DataFrame to long format
grad_rate_df = grad_rate_df.melt(id_vars=['state_abrv'], 
                    var_name='year', 
                    value_name='grad_rate')

# Convert 'year' to int and 'grad_rate' to float
grad_rate_df['year'] = pd.to_numeric(grad_rate_df['year'], errors='coerce')
grad_rate_df['grad_rate'] = pd.to_numeric(grad_rate_df['grad_rate'], errors='coerce')
grad_rate_df = grad_rate_df.sort_values(by=['state_abrv','year']).reset_index(drop=True)
# print(grad_rate_df.to_string())

ref = data_pull.get_collected_data(metric='hs_grad_rate (ref)')
ref = ref.iloc[:,:14]
ref.columns = ['state', 'state_abrv']+ [x.split('-')[-1].strip() for x in ref.columns[2:]]
# print(ref.to_string())

# Assuming you already have the original wide-format DataFrame called `ref`
# First, replace '---' strings with NaN and ensure all values from 2011-2023 are numeric
ref.replace('---', pd.NA, inplace=True)

# Convert all year columns to numeric (safe conversion)
for year in ref.columns[2:]:
    ref[year] = pd.to_numeric(ref[year], errors='coerce')

# Now melt the DataFrame from wide to long format
ref_long = ref.melt(id_vars=['state', 'state_abrv'], 
                  var_name='year', 
                  value_name='grad_rate')

# Optional: Convert 'year' column to int (if all values are valid years)
ref_long['year'] = pd.to_numeric(ref_long['year'], errors='coerce')

# Sort and reset index for cleaner output
ref_long = ref_long.loc[:,["state_abrv", "year", "grad_rate"]]
ref_long = ref_long.sort_values(by=['state_abrv', 'year']).reset_index(drop=True)

print(ref_long.to_string())


    state_abrv  year  grad_rate
0           AK  2011      68.00
1           AK  2012      70.00
2           AK  2013      71.80
3           AK  2014      71.10
4           AK  2015      75.60
5           AK  2016      76.10
6           AK  2017      78.20
7           AK  2018      78.50
8           AK  2019      80.40
9           AK  2020      79.00
10          AK  2021      78.00
11          AK  2022      78.00
12          AL  2011      72.00
13          AL  2012      75.00
14          AL  2013      80.00
15          AL  2014      86.30
16          AL  2015      89.30
17          AL  2016      87.10
18          AL  2017      89.30
19          AL  2018      90.00
20          AL  2019      91.70
21          AL  2020      91.00
22          AL  2021      91.00
23          AL  2022      91.00
24          AR  2011      81.00
25          AR  2012      84.00
26          AR  2013      84.90
27          AR  2014      86.90
28          AR  2015      84.90
29          AR  2016      87.00
30      

In [37]:
full_grad_rate_df = pd.concat([grad_rate_df,ref_long]).sort_values(by=['state_abrv','year']).reset_index(drop=True)
# print(full_grad_rate_df.to_string())

In [38]:
# GRAPHS overall grad rate
g_width = 3375
g_height = 691
us_only = full_grad_rate_df[full_grad_rate_df['state_abrv']=='US'].reset_index(drop=True)
offset = 1
filename = f'7.1.png'
for state in state_abbreviations_priority:
    if state == 'US':
        continue
    result = full_grad_rate_df[full_grad_rate_df['state_abrv']==state].reset_index(drop=True)
    print(result.to_string())
    fig = graph_grad_rate(result,us_only, 'grad_rate', state, get_col_uniq_vals(result['year']), offset)
    fig.update_layout(
        width=g_width,
        height=g_height,
        font=dict(
            family='Lato',
            size=22,
            color=hunt_darkgray
        )
    )

    if show_it:
        fig.show()
        if just_one:
            break
    if save_it:
        save_to_folder(fig,filename,g_width,g_height,juri)

    save_to_folder(fig,filename,g_width,g_height,state)
    # fig.show()
    # break
    


   state_abrv  year  grad_rate
0          AZ  2011      78.00
1          AZ  2012      76.00
2          AZ  2013      75.10
3          AZ  2014      75.70
4          AZ  2015      77.40
5          AZ  2016      79.50
6          AZ  2017      78.00
7          AZ  2018      78.70
8          AZ  2019      77.80
9          AZ  2020      77.00
10         AZ  2021      78.00
11         AZ  2022      76.00
12         AZ  2023      76.99
13         AZ  2024      78.40


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




   state_abrv  year  grad_rate
0          CT  2011       83.0
1          CT  2012       85.0
2          CT  2013       85.5
3          CT  2014       87.0
4          CT  2015       87.2
5          CT  2016       87.4
6          CT  2017       87.9
7          CT  2018       88.4
8          CT  2019       88.5
9          CT  2020       88.0
10         CT  2021       90.0
11         CT  2022       89.0
12         CT  2023       88.4
13         CT  2024       88.9


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




   state_abrv  year  grad_rate
0          DC  2011      59.00
1          DC  2012      59.00
2          DC  2013      62.30
3          DC  2014      61.40
4          DC  2015      68.50
5          DC  2016      69.20
6          DC  2017      73.20
7          DC  2018      68.50
8          DC  2019      68.90
9          DC  2020      70.90
10         DC  2021      72.63
11         DC  2022      74.86
12         DC  2023      76.10
13         DC  2024      76.10


...

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




   state_abrv  year  grad_rate
0          WV  2011       78.0
1          WV  2012       79.0
2          WV  2013       81.4
3          WV  2014       84.5
4          WV  2015       86.5
5          WV  2016       89.8
6          WV  2017       89.4
7          WV  2018       90.2
8          WV  2019       91.3
9          WV  2020       92.0
10         WV  2021       91.0
11         WV  2022       91.0
12         WV  2023       93.0
13         WV  2024       92.6


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




   state_abrv  year  grad_rate
0          WY  2011       80.0
1          WY  2012       79.0
2          WY  2013       77.0
3          WY  2014       78.6
4          WY  2015       79.3
5          WY  2016       80.0
6          WY  2017       86.2
7          WY  2018       81.7
8          WY  2019       82.1
9          WY  2020       82.1
10         WY  2021       82.0
11         WY  2022       82.0
12         WY  2023       81.0
13         WY  2024       81.6


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




In [39]:
# GRAPHS Grad Rate By Race and Ethnicity
g_width = 2700
g_height = 728
filename = f'7.2.png'
grad_rate_by_re = data_pull.get_collected_data(metric='hs_grad_rate_subgroups')
# print(grad_rate_by_re.to_string())

# grad_rate_by_re.columns = ['state_abrv','priority','total', including]

import pandas as pd
# Assuming your DataFrame is called df3
# Example cleanup before melting
grad_rate_by_re = grad_rate_by_re.dropna(subset=['state_abrv', 'updated year'])  # Drop rows missing essential data

# List of columns that contain graduation rate values by group
group_columns = [
    'Total',
    'American Indian / Alaska Native',
    'Asian/Pacific Islander',
    'Hispanic',
    'Black',
    'White',
    'Two or more races'
]

# Melt to long format
grad_rate_by_re = grad_rate_by_re.melt(
    id_vars=['state_abrv', 'updated year'],
    value_vars=group_columns,
    var_name='group',
    value_name='grad_rate'
)
grad_rate_by_re['group'] = grad_rate_by_re['group'].replace({
    'Two or more races': 'Two or More'
})

# Rename for consistency
grad_rate_by_re.rename(columns={'updated year': 'year'}, inplace=True)

# # Drop rows without graduation rate values
# grad_rate_by_re = grad_rate_by_re.dropna(subset=['grad_rate'])

# Optional: convert year to int and grad_rate to float
grad_rate_by_re['year'] = pd.to_numeric(grad_rate_by_re['year'], errors='coerce').astype('Int64')
grad_rate_by_re['grad_rate'] = pd.to_numeric(grad_rate_by_re['grad_rate'], errors='coerce')

# Reset index
grad_rate_by_re = grad_rate_by_re[grad_rate_by_re['state_abrv']!='x'].reset_index(drop=True)

# Preview result
# print(grad_rate_by_re.to_string())

us_only = grad_rate_by_re[grad_rate_by_re['state_abrv']=='US'].reset_index(drop=True)
print(us_only.to_string())
for state in state_abbreviations_priority:
    if state == 'US':
        continue
    result = grad_rate_by_re[grad_rate_by_re['state_abrv']==state].reset_index(drop=True)
    print(result.to_string())
    fig = graph_gradrate_re(result, us_only, state)
    fig.update_layout(
        width=g_width,
        height=g_height,
        font=dict(
            family='Lato',
            size=22,
            color=hunt_darkgray),  
        )
    
    fig.update_yaxes(
           visible=False
        )
    

    if show_it:
        fig.show()
        if just_one:
            break
    if save_it:
        save_to_folder(fig,filename,g_width,g_height,state)





  state_abrv  year                            group  grad_rate
0         US  2022                            Total       87.0
1         US  2022  American Indian / Alaska Native       74.0
2         US  2022           Asian/Pacific Islander       93.0
3         US  2022                         Hispanic       82.0
4         US  2022                            Black       81.0
5         US  2022                            White       90.0
6         US  2022                      Two or More        NaN
  state_abrv  year                            group  grad_rate
0         AZ  2024                            Total      77.86
1         AZ  2024  American Indian / Alaska Native      67.54
2         AZ  2024           Asian/Pacific Islander        NaN
3         AZ  2024                         Hispanic      74.94
4         AZ  2024                            Black      72.14
5         AZ  2024                            White      84.07
6         AZ  2024                      Two or More    

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




  state_abrv  year                            group  grad_rate
0         CT  2024                            Total      88.90
1         CT  2024  American Indian / Alaska Native      83.20
2         CT  2024           Asian/Pacific Islander      95.36
3         CT  2024                         Hispanic      82.30
4         CT  2024                            Black      83.70
5         CT  2024                            White      93.70
6         CT  2024                      Two or More      90.10
Original df groups: ['Total' 'American Indian / Alaska Native' 'Asian/Pacific Islander'
 'Hispanic' 'Black' 'White' 'Two or More']
Original us_df groups: ['Total' 'American Indian/Alaska Native' 'Asian/Pacific Islander'
 'Hispanic' 'Black' 'White' 'Two or More']
After mapping df groups: ['Total' 'American Indian/Alaska Native' 'Asian/Pacific Islander'
 'Hispanic' 'Black' 'White' 'Two or More']
After mapping us_df groups: ['Total' 'American Indian/Alaska Native' 'Asian/Pacific Islander'
 'His

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




  state_abrv  year                            group  grad_rate
0         DC  2024                            Total      76.77
1         DC  2024  American Indian / Alaska Native        NaN
2         DC  2024           Asian/Pacific Islander      92.31
3         DC  2024                         Hispanic      76.25
4         DC  2024                            Black      73.18
5         DC  2024                            White      96.14
6         DC  2024                      Two or More      87.04
Original df groups: ['Total' 'American Indian / Alaska Native' 'Asian/Pacific Islander'
 'Hispanic' 'Black' 'White' 'Two or More']
Original us_df groups: ['Total' 'American Indian/Alaska Native' 'Asian/Pacific Islander'
 'Hispanic' 'Black' 'White' 'Two or More']
After mapping df groups: ['Total' 'American Indian/Alaska Native' 'Asian/Pacific Islander'
 'Hispanic' 'Black' 'White' 'Two or More']
After mapping us_df groups: ['Total' 'American Indian/Alaska Native' 'Asian/Pacific Islander'
 'His

...

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




  state_abrv  year                            group  grad_rate
0         WV  2024                            Total     0.9258
1         WV  2024  American Indian / Alaska Native     1.0000
2         WV  2024           Asian/Pacific Islander        NaN
3         WV  2024                         Hispanic     0.9130
4         WV  2024                            Black     0.8890
5         WV  2024                            White     0.9280
6         WV  2024                      Two or More     0.9100
Original df groups: ['Total' 'American Indian / Alaska Native' 'Asian/Pacific Islander'
 'Hispanic' 'Black' 'White' 'Two or More']
Original us_df groups: ['Total' 'American Indian/Alaska Native' 'Asian/Pacific Islander'
 'Hispanic' 'Black' 'White' 'Two or More']
After mapping df groups: ['Total' 'American Indian/Alaska Native' 'Asian/Pacific Islander'
 'Hispanic' 'Black' 'White' 'Two or More']
After mapping us_df groups: ['Total' 'American Indian/Alaska Native' 'Asian/Pacific Islander'
 'His

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




  state_abrv  year                            group  grad_rate
0         WY  2024                            Total   0.816000
1         WY  2024  American Indian / Alaska Native   0.528000
2         WY  2024           Asian/Pacific Islander   0.797297
3         WY  2024                         Hispanic   0.781000
4         WY  2024                            Black   0.785000
5         WY  2024                            White   0.837000
6         WY  2024                      Two or More   0.768000
Original df groups: ['Total' 'American Indian / Alaska Native' 'Asian/Pacific Islander'
 'Hispanic' 'Black' 'White' 'Two or More']
Original us_df groups: ['Total' 'American Indian/Alaska Native' 'Asian/Pacific Islander'
 'Hispanic' 'Black' 'White' 'Two or More']
After mapping df groups: ['Total' 'American Indian/Alaska Native' 'Asian/Pacific Islander'
 'Hispanic' 'Black' 'White' 'Two or More']
After mapping us_df groups: ['Total' 'American Indian/Alaska Native' 'Asian/Pacific Islander'
 'His

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




In [40]:
# GRAPHS Grad Rate By Other Subgroups

g_width = 2704
g_height = 651
filename = f'7.3.png'

grad_rate_by_re = data_pull.get_collected_data(metric='hs_grad_rate_subgroups')
# print(grad_rate_by_re.to_string())

# grad_rate_by_re.columns = ['state_abrv','priority','total', including]

import pandas as pd

# Assuming your DataFrame is called df3
# Example cleanup before melting
grad_rate_by_re = grad_rate_by_re.dropna(subset=['state_abrv', 'updated year'])  # Drop rows missing essential data

# List of columns that contain graduation rate values by group
group_columns = [
    'Economically disadvantaged (Based on state definition)',
    'Limited English proficiency',
    'Students with disabilities'
]


# Melt to long format
grad_rate_by_re = grad_rate_by_re.melt(
    id_vars=['state_abrv', 'updated year'],
    value_vars=group_columns,
    var_name='group',
    value_name='grad_rate'
)
grad_rate_by_re['group'] = grad_rate_by_re['group'].replace({
    'Two or more races': 'Two or More'
})


# Rename for consistency
grad_rate_by_re.rename(columns={'updated year': 'year'}, inplace=True)

# # Drop rows without graduation rate values
# grad_rate_by_re = grad_rate_by_re.dropna(subset=['grad_rate'])

# Optional: convert year to int and grad_rate to float
grad_rate_by_re['year'] = pd.to_numeric(grad_rate_by_re['year'], errors='coerce').astype('Int64')
grad_rate_by_re['grad_rate'] = pd.to_numeric(grad_rate_by_re['grad_rate'], errors='coerce')

# Reset index
grad_rate_by_re = grad_rate_by_re[grad_rate_by_re['state_abrv']!='x'].reset_index(drop=True)

# Preview result
# print(grad_rate_by_re.to_string())

us_only = grad_rate_by_re[grad_rate_by_re['state_abrv']=='US'].reset_index(drop=True)
# print(us_only.to_string())
for state in state_abbreviations_priority:
    if state == 'US':
        continue
    result = grad_rate_by_re[grad_rate_by_re['state_abrv']==state].reset_index(drop=True)
    # print(result.to_string())
    fig = graph_gradrate_other_subgroup(group_columns,result, us_only, state)
    fig.update_layout(
        width = 2704,
        height = 651
    )
    fig.update_layout(
        width=g_width,
        height=g_height,
        # font=dict(
        #     family='Lato',
        #     size=1,
        #     color=hunt_darkgray)
    )
    
    fig.update_yaxes(
           visible=False
        )

    if show_it:
        fig.show()
        if just_one:
            break
    if save_it:
        save_to_folder(fig,filename,g_width,g_height,state)







C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:

## Page 8

### 8.1: Dropout Rate

In [41]:
dropout_df = data_pull.get_collected_data(metric = 'dropout_rate')
dropout_df = dropout_df.iloc[:,:11]
print(dropout_df.to_string(max_colwidth=30))


# Step 1: Rename columns properly
dropout_df.columns = [
    'state_abrv', 'priority', 'source', 'updated_year', 'Total',
    'White', 'Black', 'Hispanic', 'Asian/Pacific Islander',
    'American Indian/Alaska Native', 'Two or more races'
]

# Step 2: Keep only rows where state_abrv is not NaN or 'x'
df_clean = dropout_df[~dropout_df['state_abrv'].isna()]
df_clean = df_clean[df_clean['state_abrv'] != 'x']

# Step 3: Select columns of interest
group_columns = [
    'Total', 'White', 'Black', 'Hispanic', 'Asian/Pacific Islander',
    'American Indian/Alaska Native', 'Two or more races'
]
df_clean = df_clean[['state_abrv', 'updated_year'] + group_columns]

# Step 4: Convert updated_year to a single year (pick later if range)
def extract_later_year(val):
    if pd.isna(val):
        return np.nan
    # Match years in format "YYYY" or "YYYY-YYYY"
    years = re.findall(r'\d{4}', str(val))
    if not years:
        return np.nan
    return int(years[-1])  # pick the later year

df_clean['updated_year'] = df_clean['updated_year'].apply(extract_later_year)

# Step 5: Melt to long format
df_long = df_clean.melt(
    id_vars=['state_abrv', 'updated_year'],
    value_vars=group_columns,
    var_name='group',
    value_name='value'
)

# Step 6: Convert values to float and standardize percentages
def normalize_percentage(x):
    try:
        x = float(x)
        if x > 1.5:  # Likely stored as 35.1 → treat as 35.1%
            return x
        else:        # Already decimal (0.351 → 35.1%)
            return x * 100
    except:
        return np.nan

df_long['value'] = df_long['value'].apply(normalize_percentage)

# Step 7: Rename column
df_long.rename(columns={'updated_year': 'year'}, inplace=True)

# Optional: sort for readability
dropout_df = df_long.sort_values(['state_abrv', 'group']).reset_index(drop=True)




   Metric: 2023-2024 Dropout Rate Unnamed: 1                     Unnamed: 2         Unnamed: 3 Unnamed: 4 Unnamed: 5 Unnamed: 6 Unnamed: 7              Unnamed: 8                     Unnamed: 9        Unnamed: 10
0                           state  priority?            updated source link  updated data year      Total      White      Black   Hispanic  Asian/pacific islander  American Indian/Alaska Native  Two or more races
1                              AL        NaN  https://nces.ed.gov/progra...          2022-2023      0.017      0.012      0.023      0.026                   0.051                          0.015              0.018
2                              AK        NaN  https://education.alaska.g...          2023-2024     0.0356      0.023      0.035      0.032                    0.03                          0.065              0.038
3                              US          Y  https://nces.ed.gov/progra...          2022-2023      0.028       0.02       0.04      0.036          

In [42]:
# GRAPHS Dropout rate by Race and Ethnicity
g_width = 4036
g_height = 763
filename = f'8.1.png'
# Reset index

# Preview result
# print(grad_rate_by_re.to_string())

us_only = dropout_df[dropout_df['state_abrv']=='US'].reset_index(drop=True)
print(us_only.to_string())
for state in state_abbreviations_priority:
    if state == 'US':
        continue
    result = dropout_df[dropout_df['state_abrv']==state].reset_index(drop=True)
    print(result.to_string())
    fig = graph_dropouts(result, us_only, state)
    fig.update_layout(
        width=g_width,
        height=g_height,
        font=dict(
            family='Lato',
            size=22,
            color=hunt_darkgray),  
        )
    
    fig.update_yaxes(
           visible=False
        )
    save_to_folder(fig,filename,g_width,g_height,state)
    # fig.show()
    # break






  state_abrv    year                          group  value
0         US  2023.0  American Indian/Alaska Native    6.0
1         US  2023.0         Asian/Pacific Islander    6.1
2         US  2023.0                          Black    4.0
3         US  2023.0                       Hispanic    3.6
4         US  2023.0                          Total    2.8
5         US  2023.0              Two or more races    3.1
6         US  2023.0                          White    2.0
  state_abrv    year                          group  value
0         AZ  2024.0  American Indian/Alaska Native   9.07
1         AZ  2024.0         Asian/Pacific Islander    NaN
2         AZ  2024.0                          Black   6.03
3         AZ  2024.0                       Hispanic   5.81
4         AZ  2024.0                          Total   5.25
5         AZ  2024.0              Two or more races   4.36
6         AZ  2024.0                          White   3.72
Original df groups: ['American Indian/Alaska Native' 'As

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




  state_abrv    year                          group  value
0         CT  2023.0  American Indian/Alaska Native    1.6
1         CT  2023.0         Asian/Pacific Islander    1.7
2         CT  2023.0                          Black    2.2
3         CT  2023.0                       Hispanic    2.8
4         CT  2023.0                          Total    1.5
5         CT  2023.0              Two or more races    1.2
6         CT  2023.0                          White    0.7
Original df groups: ['American Indian/Alaska Native' 'Asian/Pacific Islander' 'Black'
 'Hispanic' 'Total' 'Two or more races' 'White']
Original us_df groups: ['American Indian/Alaska Native' 'Asian/Pacific Islander' 'Black'
 'Hispanic' 'Total' 'Two or More' 'White']
After mapping df groups: ['American Indian/Alaska Native' 'Asian/Pacific Islander' 'Black'
 'Hispanic' 'Total' 'Two or More' 'White']
After mapping us_df groups: ['American Indian/Alaska Native' 'Asian/Pacific Islander' 'Black'
 'Hispanic' 'Total' 'Two or More'

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




  state_abrv    year                          group  value
0         DC  2023.0  American Indian/Alaska Native   10.0
1         DC  2023.0         Asian/Pacific Islander   10.0
2         DC  2023.0                          Black   13.0
3         DC  2023.0                       Hispanic   10.4
4         DC  2023.0                          Total   12.2
5         DC  2023.0              Two or more races   10.0
6         DC  2023.0                          White   11.4
Original df groups: ['American Indian/Alaska Native' 'Asian/Pacific Islander' 'Black'
 'Hispanic' 'Total' 'Two or more races' 'White']
Original us_df groups: ['American Indian/Alaska Native' 'Asian/Pacific Islander' 'Black'
 'Hispanic' 'Total' 'Two or More' 'White']
After mapping df groups: ['American Indian/Alaska Native' 'Asian/Pacific Islander' 'Black'
 'Hispanic' 'Total' 'Two or More' 'White']
After mapping us_df groups: ['American Indian/Alaska Native' 'Asian/Pacific Islander' 'Black'
 'Hispanic' 'Total' 'Two or More'

...

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




  state_abrv    year                          group  value
0         WV  2023.0  American Indian/Alaska Native    0.0
1         WV  2023.0         Asian/Pacific Islander    2.3
2         WV  2023.0                          Black    1.1
3         WV  2023.0                       Hispanic    0.9
4         WV  2023.0                          Total    0.8
5         WV  2023.0              Two or more races    0.8
6         WV  2023.0                          White    0.8
Original df groups: ['American Indian/Alaska Native' 'Asian/Pacific Islander' 'Black'
 'Hispanic' 'Total' 'Two or more races' 'White']
Original us_df groups: ['American Indian/Alaska Native' 'Asian/Pacific Islander' 'Black'
 'Hispanic' 'Total' 'Two or More' 'White']
After mapping df groups: ['American Indian/Alaska Native' 'Asian/Pacific Islander' 'Black'
 'Hispanic' 'Total' 'Two or More' 'White']
After mapping us_df groups: ['American Indian/Alaska Native' 'Asian/Pacific Islander' 'Black'
 'Hispanic' 'Total' 'Two or More'

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




  state_abrv    year                          group  value
0         WY  2023.0  American Indian/Alaska Native   13.1
1         WY  2023.0         Asian/Pacific Islander   13.1
2         WY  2023.0                          Black    4.6
3         WY  2023.0                       Hispanic    4.4
4         WY  2023.0                          Total    3.9
5         WY  2023.0              Two or more races    5.5
6         WY  2023.0                          White    3.4
Original df groups: ['American Indian/Alaska Native' 'Asian/Pacific Islander' 'Black'
 'Hispanic' 'Total' 'Two or more races' 'White']
Original us_df groups: ['American Indian/Alaska Native' 'Asian/Pacific Islander' 'Black'
 'Hispanic' 'Total' 'Two or More' 'White']
After mapping df groups: ['American Indian/Alaska Native' 'Asian/Pacific Islander' 'Black'
 'Hispanic' 'Total' 'Two or More' 'White']
After mapping us_df groups: ['American Indian/Alaska Native' 'Asian/Pacific Islander' 'Black'
 'Hispanic' 'Total' 'Two or More'

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




### 8.2: CTE concentratos

In [43]:
# GRAPHS
cte_df = data_pull.cte_calculated()
# print(cte_df.to_string())

# GRAPHS Grad Rate By Race and Ethnicity
g_width = 2482
g_height = 660
filename = f'8.2.png'
grad_rate_by_re = data_pull.get_collected_data(metric='hs_grad_rate_subgroups')

for state in state_abbreviations_priority:
    
    if state == 'US':
        continue
    fig = graph_8_cte(cte_df,state)
    fig.update_yaxes(
        visible=False
    )
    fig.update_layout(
    width=g_width,
    height=g_height,
    font=dict(
        family='Lato',
        size=22,
        color=hunt_darkgray),  
    )
    save_to_folder(fig,filename,g_width,g_height,state)
    # fig.show()
    # break




c:\Users\clutz\Envs\hunt_env\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning:

Workbook contains no default style, apply openpyxl's default



  state                   group perc_o_cte perc_o_enr  order
0    AZ            nat_am_or_ak   0.043333   0.115965      1
5    AZ  asian_pacific_islander   0.034409   0.112941      2
2    AZ                hispanic   0.458746   0.108931      3
1    AZ                   black   0.044982   0.089482      4
3    AZ                   white   0.381746   0.123628      5
4    AZ             two_or_more   0.036784   0.099985      6


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




  state                   group perc_o_cte perc_o_enr  order
0    CT            nat_am_or_ak   0.002475   0.239283      1
5    CT  asian_pacific_islander   0.053181   0.244713      2
2    CT                hispanic   0.274242   0.220765      3
1    CT                   black   0.122907   0.236631      4
3    CT                   white   0.510719    0.26029      5
4    CT             two_or_more   0.036475   0.197511      6


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




...

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




  state                   group perc_o_cte perc_o_enr  order
0    WV            nat_am_or_ak   0.000908   0.259067      1
5    WV  asian_pacific_islander   0.006595   0.204277      2
2    WV                hispanic   0.022439   0.223854      3
1    WV                   black   0.041989   0.224784      4
3    WV                   white   0.890167   0.220081      5
4    WV             two_or_more   0.037901   0.192382      6


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




  state                   group perc_o_cte perc_o_enr  order
0    WY            nat_am_or_ak   0.022389   0.159972      1
5    WY  asian_pacific_islander   0.008529   0.237517      2
2    WY                hispanic   0.122898    0.18669      3
1    WY                   black   0.007366   0.197659      4
3    WY                   white   0.808384   0.234924      5
4    WY             two_or_more   0.030434   0.180512      6


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




### AP graphs 


In [44]:
# GRAPHS participation

sat_df = data_pull.get_sat_graph_data()
# print(sat_df.to_string())

g_width = 901
g_height = 499
filename = f'8.3.png'

for state in state_abbreviations_priority:
    if state =="US":
        continue
    us_only = sat_df[sat_df['state_abrv']=="US"]['perc_in_an_ap'].to_list()
    state_only = sat_df[sat_df['state_abrv']==state]['perc_in_an_ap'].to_list()

    print(us_only)
    print(state_only)
    
    fig = graph_ap(us_only[0],state_only[0],state, 'percent')
    fig.update_layout(
        width=g_width,
        height=g_height,
        font=dict(
            family='Lato',
            size=22,
            color=hunt_darkgray),  
        )
    
    fig.update_yaxes(
           visible=False
        )
    
    if show_it:
        fig.show()
        if just_one:
            break
    if save_it:
        save_to_folder(fig,filename,g_width,g_height,state)


    # break


[0.3532730189747818]
[0.2336975426104393]
[0.3532730189747818]
[0.40353496541881656]


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




[0.3532730189747818]
[0.609008692598121]
[0.3532730189747818]
[0.4024127724459593]
[0.3532730189747818]
[0.42144807973342047]
[0.3532730189747818]
[0.16040584834088314]
[0.3532730189747818]
[0.4393563111435254]
[0.3532730189747818]
[0.30095915293283715]
[0.3532730189747818]
[0.15890200708382526]
[0.3532730189747818]
[0.14057943330149633]
[0.3532730189747818]
[0.31144162369053385]
[0.3532730189747818]
[0.47627888703718957]
[0.3532730189747818]
[0.4393942099905345]
[0.3532730189747818]
[0.29664427284211936]
[0.3532730189747818]
[0.28020790932147666]
[0.3532730189747818]
[0.19078858769887838]
[0.3532730189747818]
[0.19451546688753646]
[0.3532730189747818]
[0.385936068474755]
[0.3532730189747818]
[0.24227682840630327]
[0.3532730189747818]
[0.43892415219484365]
[0.3532730189747818]
[0.3792310260314462]
[0.3532730189747818]
[0.18114632944654294]
[0.3532730189747818]
[0.2782783392183124]
[0.3532730189747818]
[0.17549999003010908]
[0.3532730189747818]
[0.22135351633786043]
[0.3532730189747818]

In [45]:
# GRAPHS num ap exams per 1000
sat_df = data_pull.get_sat_graph_data()
# print(sat_df.to_string())

g_width = 945
g_height = 500
filename = f'8.4.png'


for state in state_abbreviations_priority:
    if state =="US":
        continue
    us_only = sat_df[sat_df['state_abrv']=="US"]['ap_exams_per_k_11_12'].to_list()
    state_only = sat_df[sat_df['state_abrv']==state]['ap_exams_per_k_11_12'].to_list()

    print(us_only)
    print(state_only)
    
    fig = graph_ap(us_only[0],state_only[0],state, 'number')
    fig.update_layout(
        width=g_width,
        height=g_height,
        font=dict(
            family='Lato',
            size=22,
            color=hunt_darkgray),  
        )
    
    fig.update_yaxes(
           visible=False
        )

    if show_it:
        fig.show()
        if just_one:
            break
    if save_it:
        save_to_folder(fig,filename,g_width,g_height,state)

    # break


[500.371745167851]
[323.85979988951]


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




[500.371745167851]
[655.08837413547]
[500.371745167851]
[993.414698393186]
[500.371745167851]
[541.842317696654]
[500.371745167851]
[554.302050813901]
[500.371745167851]
[220.951644385519]
[500.371745167851]
[625.462113019641]
[500.371745167851]
[418.078814460393]
[500.371745167851]
[179.374262101535]
[500.371745167851]
[192.346386501114]
[500.371745167851]
[314.25288011321]
[500.371745167851]
[652.899317546717]
[500.371745167851]
[770.899933920311]
[500.371745167851]
[422.582345411497]
[500.371745167851]
[319.17838244043]
[500.371745167851]
[255.82767399655]
[500.371745167851]
[223.108502286723]
[500.371745167851]
[701.452076456783]
[500.371745167851]
[322.377447923163]
[500.371745167851]
[660.93365545769]
[500.371745167851]
[515.075765594847]
[500.371745167851]
[205.780916406635]
[500.371745167851]
[400.961128698405]
[500.371745167851]
[235.179757133457]
[500.371745167851]
[281.931344487773]
[500.371745167851]
[559.280098966439]
[500.371745167851]
[318.718750790452]
[500.371745167851

In [46]:
#GRAPHS Percent passing
sat_df = data_pull.get_sat_graph_data()
# print(sat_df.to_string())
g_width = 1471
g_height = 500
filename = f'8.5.png'


for state in state_abbreviations_priority:
    if state =="US":
        continue
    us_only = sat_df[sat_df['state_abrv']=="US"]['perc_3_orbetter'].to_list()
    state_only = sat_df[sat_df['state_abrv']==state]['perc_3_orbetter'].to_list()

    print(us_only)
    print(state_only)
    
    fig = graph_ap(us_only[0],state_only[0],state, 'percent')
    
    fig.update_layout(
        width=g_width,
        height=g_height,
        font=dict(
            family='Lato',
            size=22,
            color=hunt_darkgray),  
        )
    
    fig.update_yaxes(
           visible=False
        )
    

    if show_it:
        fig.show()
        if just_one:
            break
    if save_it:
        save_to_folder(fig,filename,g_width,g_height,state)


[0.660227674346602]
[0.65869880306714]
[0.660227674346602]
[0.7297262817696]


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




[0.660227674346602]
[0.613563983786914]
[0.660227674346602]
[0.614705329742594]
[0.660227674346602]
[0.695809648372655]
[0.660227674346602]
[0.687068727573834]
[0.660227674346602]
[0.680662809461959]
[0.660227674346602]
[0.624926166568222]
[0.660227674346602]
[0.703490620769677]
[0.660227674346602]
[0.738794686251215]
[0.660227674346602]
[0.614090770868177]
[0.660227674346602]
[0.67362268073533]
[0.660227674346602]
[0.71342458272178]
[0.660227674346602]
[0.698309928868523]
[0.660227674346602]
[0.724910892162576]
[0.660227674346602]
[0.700377072401891]
[0.660227674346602]
[0.662761314036297]
[0.660227674346602]
[0.730676202043893]
[0.660227674346602]
[0.45445397733291]
[0.660227674346602]
[0.677117444201979]
[0.660227674346602]
[0.668019489825165]
[0.660227674346602]
[0.695274636510501]
[0.660227674346602]
[0.740241196926313]
[0.660227674346602]
[0.567214765100671]
[0.660227674346602]
[0.674992129289537]
[0.660227674346602]
[0.642339511319182]
[0.660227674346602]
[0.707351320607159]
[0.

## Page 9

In [47]:
# College Entrance Exams
g_width = 2482
g_height = 660
filename = f'9.1.png'

#data set up
act_df = data_pull.get_collected_data(metric='act_benchmarks', no_header=True)
act_df = act_df.iloc[1:,1:].reset_index(drop = True)
act_df.columns = act_df.iloc[0,:]

act_df = act_df.iloc[1:,:].reset_index(drop=True)
print(act_df.to_string())


us_only = act_df[act_df['State']=='US']
print(us_only.to_string())
for state in state_abbreviations_priority:
    result = act_df[act_df['State']==state]
    print(result.to_string())
    fig = graph_act_benchmarks(result, us_only, state)
    fig.update_yaxes(
        visible=False
    )
    fig.update_layout(
    width=g_width,
    height=g_height,
    font=dict(
        family='Lato',
        size=22,
        color=hunt_darkgray),  
    )

    if show_it:
        fig.show()
        if just_one:
            break
    if save_it:
        save_to_folder(fig,filename,g_width,g_height,state)
    



0  State Total Black American Indian/Alaska Native White Hispanic Asian Native Hawaiian/Other PI Two or more races
0     US   0.3  0.09                           0.1  0.39     0.17   0.6                     0.11              0.31
1     AL   0.2  0.06                          0.12  0.29     0.12  0.53                      0.1              0.22
2     AK  0.35  0.23                          0.02  0.56     0.33  0.29                        0              0.39
3     AZ  0.21  0.09                          0.05  0.34     0.11  0.57                     0.09              0.31
4     AR  0.23  0.06                          0.15   0.3     0.14   0.5                     0.03              0.24
5     CA  0.75   0.3                          0.48   0.8      0.5   0.9                     0.61              0.81
6     CO  0.66  0.49                          0.47  0.69     0.46  0.81                      0.5              0.72
7     CT  0.79  0.57                             1   0.8      0.7  0.88         

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




0 State Total Black American Indian/Alaska Native White Hispanic Asian Native Hawaiian/Other PI Two or more races
7    CT  0.79  0.57                             1   0.8      0.7  0.88                        1              0.83
['Total', 'Black', 'American Indian/Alaska Native', 'White', 'Hispanic', 'Asian', 'Native Hawaiian/Other PI', 'Two or more races']


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




0 State Total Black American Indian/Alaska Native White Hispanic Asian Native Hawaiian/Other PI Two or more races
9    DC  0.79  0.37                             1  0.86      0.8  0.86                        0              0.79
['Total', 'Black', 'American Indian/Alaska Native', 'White', 'Hispanic', 'Asian', 'Native Hawaiian/Other PI', 'Two or more races']


...

C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




0  State Total Black American Indian/Alaska Native White Hispanic Asian Native Hawaiian/Other PI Two or more races
49    WV   0.3  0.16                             0   0.3     0.32   0.5                     0.33              0.32
['Total', 'Black', 'American Indian/Alaska Native', 'White', 'Hispanic', 'Asian', 'Native Hawaiian/Other PI', 'Two or more races']


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




0  State Total Black American Indian/Alaska Native White Hispanic Asian Native Hawaiian/Other PI Two or more races
51    WY  0.28  0.09                          0.06  0.32     0.15  0.39                        0              0.29
['Total', 'Black', 'American Indian/Alaska Native', 'White', 'Hispanic', 'Asian', 'Native Hawaiian/Other PI', 'Two or more races']


C:\Users\clutz\AppData\Local\Temp\ipykernel_23580\2484680588.py:8: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


